# Project 15 — QAOA Max-Cut on IonQ hardware: an application-engineering study
**Level:** Advanced · **Vendor target:** IonQ (trapped-ion, all-to-all) · **Core:** NumPy + stdlib, in-process.

> **What this notebook is for.** It is the artefact a quantum pre-sales engineer would hand a
> customer: a *named business problem*, the *QUBO and cost Hamiltonian* derived from it, three
> tiers of evidence (our own ideal simulator, our own noisy simulator driven by IonQ's published
> error figures, and a real IonQ QPU job when credentials are present), measured *quality / depth /
> shots / cost / latency*, and a one-page report that says plainly what the hardware achieves, what
> it does not achieve yet, and whether it beats a classical solver. Sections that depend on vendor
> numbers are marked and carry a source and a date; sections that depend on our own code print the
> number they measured.

## Part 0 — Setup

In [43]:
import itertools
import json
import math
import os
import sys
import time

import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

plt.rcParams.update({"figure.dpi": 100, "savefig.dpi": 100, "font.size": 9,
                     "axes.grid": True, "grid.alpha": 0.25, "axes.axisbelow": True})

CHECKS = []


def check(name, cond, detail=""):
    """Record a verification. A check that cannot fail is not a check: pass a measured value."""
    ok = bool(cond)
    CHECKS.append((name, ok, detail))
    if not ok:
        print("  [FAILED] %s  %s" % (name, detail))
    return ok


print("python", sys.version.split()[0], "| numpy", np.__version__)
print("Every figure is drawn from a variable computed in this session; nothing is hard-painted.")


python 3.13.18 | numpy 2.4.6
Every figure is drawn from a variable computed in this session; nothing is hard-painted.


---

## Part 1 — Deliverable 1: the business problem, stated the way the customer states it

**The customer.** A regional last-mile courier operator runs one depot with **8 driving crews**
(near-bay crews, so we can solve the problem exactly and say so). Each crew occupies a loading bay
in the morning or the evening, and the depot wants **two shifts** because a third shift costs
premium pay.

**The pain.** Some pairs of crews share equipment: the same pallet jack, the same refrigerated bay,
the same dock door. When both crews are on the *same* shift they contend, and the contention is
measured in lost driver-minutes per day. The operations manager has already counted these pairs —
that count is the weight `w_ij` below.

**The decision.** Put every crew on shift A or shift B so that total daily contention is minimised.

**Why this is Max-Cut, not a vague analogy.** Build a graph with one vertex per crew and one weighted
edge per contention pair. Split the vertices into two sets (the two shifts). An edge whose endpoints
land in **different** sets is *cut*, and a cut edge means that contention never happens. An edge left
*uncut* is contention we still pay for. Therefore:

```
contention(x) = sum of w_ij over uncut edges  =  W_total  -  cut(x)
minimise contention   <=>   maximise cut      <=>   Max-Cut
```

No slack in that chain: the objective is the same function with the same argmax, so any Max-Cut
result transfers directly into driver-minutes. We will keep `W_total` in view the whole time, because
"the quantum computer found a cut of 47" is meaningless to an operations manager until you also say
which contention-minutes that removes. On the instance below the total exposure is 72 min/day, the
rota in use today recovers 38 of them, and the best possible split recovers 68.

**Reading the data below:** weights are contention-minutes per day, so they are small integers — the
kind of number an operations manager actually produces from a spreadsheet, not a synthetic unit
weight.

In [44]:
CREWS = ["Alpha", "Bravo", "Charlie", "Delta", "Echo", "Foxtrot", "Golf", "Hotel"]
N = len(CREWS)

# (i, j, contention-minutes-per-day). The pairs the operator counted by hand.
CONFLICTS = [(0, 1, 2), (0, 2, 6), (1, 2, 6), (1, 3, 2), (2, 4, 9), (3, 4, 9),
             (3, 5, 8), (4, 6, 1), (5, 6, 1), (6, 7, 4), (5, 7, 3), (0, 7, 4),
             (2, 6, 8), (4, 7, 9)]

EDGES = [(i, j) for i, j, _ in CONFLICTS]
W = np.zeros((N, N))
for i, j, w in CONFLICTS:
    W[i, j] = W[j, i] = w
DEG = W.sum(axis=1)                     # d_i = total contention this crew is involved in
W_TOTAL = float(sum(w for _, _, w in CONFLICTS))

DEGREE = [int(v) for v in DEG]
print("crews             :", N)
print("contention pairs  :", len(CONFLICTS))
print("W_total (min/day) :", W_TOTAL)
print("degree per crew   :", dict(zip(CREWS, DEGREE)))
print("sum of degrees    :", sum(DEGREE), "= 2 * W_total ->", sum(DEGREE) == 2 * W_TOTAL)
assert len(set(EDGES)) == len(EDGES), "one pair, one edge"
assert all(i < j for i, j in EDGES), "edges stored upper-triangular"

# The rota the depot runs today: the first four crews on the morning shift, the rest on evening.
STATUS_QUO = np.array([1, 1, 1, 1, 0, 0, 0, 0])


def cut_value(x, edges=None, w=None):
    """Total weight separated by the bipartition x (1 = shift A, 0 = shift B)."""
    edges = EDGES if edges is None else edges
    w = W if w is None else w
    return float(sum(w[i, j] for i, j in edges if x[i] != x[j]))


def contention(x):
    return W_TOTAL - cut_value(x)


def is_balanced(x):
    """Equal crew counts on both shifts (n is even here, so imbalance is even)."""
    return int(np.sum(2 * x - 1)) == 0


def show(x):
    a = [CREWS[i] for i in range(N) if x[i] == 1]
    b = [CREWS[i] for i in range(N) if x[i] == 0]
    return "A=[%s] B=[%s]" % (",".join(a) or "-", ",".join(b) or "-")


sq_cut = cut_value(STATUS_QUO)
print()
print("status-quo rota   :", show(STATUS_QUO))
print("  cut / contention:", sq_cut, "/", contention(STATUS_QUO),
      "-> the depot pays %.1f contention-minutes a day today" % contention(STATUS_QUO))


crews             : 8
contention pairs  : 14
W_total (min/day) : 72.0
degree per crew   : {'Alpha': 12, 'Bravo': 10, 'Charlie': 29, 'Delta': 19, 'Echo': 28, 'Foxtrot': 12, 'Golf': 14, 'Hotel': 20}
sum of degrees    : 144 = 2 * W_total -> True

status-quo rota   : A=[Alpha,Bravo,Charlie,Delta] B=[Echo,Foxtrot,Golf,Hotel]
  cut / contention: 38.0 / 34.0 -> the depot pays 34.0 contention-minutes a day today


### What is the best possible answer? (brute force, because `n = 8` is 256 schedules)

Being able to *exhaustively* solve the customer's actual instance is the most useful thing in this
whole notebook: every quantum and heuristic number that follows is reported as a ratio against a
**known optimum**, not against another approximation. Keep this habit when `n` is small enough, and
say out loud when it is not.

In [45]:
ALL_X = np.array(list(itertools.product((0, 1), repeat=N)))          # (2**n, n)
ALL_CUT = np.array([cut_value(x) for x in ALL_X])

OPT = int(ALL_CUT.max())
OPT_X = ALL_X[ALL_CUT == OPT]
UNIQUE = sorted(set(ALL_CUT.tolist()))
print("schedules enumerated :", len(ALL_X))
print("optimal cut          :", OPT)
print("optimal contention   :", W_TOTAL - OPT, "min/day")
print("distinct cut values  :", len(UNIQUE), UNIQUE)
print("number of optima     : %d  (= %d up to the A<->B relabelling)" % (len(OPT_X), len(OPT_X) // 2))
for x in OPT_X:
    print("   ", show(x))
assert len(OPT_X) % 2 == 0, "optima must come in complement pairs"
print()
print("status quo cut       :", sq_cut, "-> ratio %.4f of optimum" % (sq_cut / OPT))
print("average random cut   : %.4f -> ratio %.4f" % (ALL_CUT.mean(), ALL_CUT.mean() / OPT))

# How hard is this instance? A bipartite graph would be solved by inspection (cut = W_total).
def count_triangles(edges, w):
    adj = {i: set() for i in range(N)}
    for i, j in edges:
        adj[i].add(j); adj[j].add(i)
    t = 0
    for a, b, c in itertools.combinations(range(N), 3):
        if c in adj[b] and b in adj[a] and c in adj[a]:
            t += 1
    return t


tri = count_triangles(EDGES, W)
print("odd cycles present   : %d triangles -> NOT bipartite, so cut %d < W_total %d"
      % (tri, OPT, W_TOTAL))
assert tri > 0 and OPT < W_TOTAL


schedules enumerated : 256
optimal cut          : 68
optimal contention   : 4.0 min/day
distinct cut values  : 46 [0.0, 10.0, 12.0, 14.0, 15.0, 18.0, 19.0, 20.0, 21.0, 22.0, 23.0, 24.0, 25.0, 26.0, 27.0, 28.0, 29.0, 30.0, 31.0, 32.0, 33.0, 34.0, 35.0, 36.0, 37.0, 38.0, 39.0, 40.0, 41.0, 42.0, 43.0, 44.0, 45.0, 46.0, 47.0, 48.0, 49.0, 50.0, 51.0, 52.0, 53.0, 55.0, 58.0, 60.0, 62.0, 68.0]
number of optima     : 2  (= 1 up to the A<->B relabelling)
    A=[Charlie,Delta,Hotel] B=[Alpha,Bravo,Echo,Foxtrot,Golf]
    A=[Alpha,Bravo,Echo,Foxtrot,Golf] B=[Charlie,Delta,Hotel]

status quo cut       : 38.0 -> ratio 0.5588 of optimum
average random cut   : 36.0000 -> ratio 0.5294
odd cycles present   : 4 triangles -> NOT bipartite, so cut 68 < W_total 72


### Figure 1 — the instance, and the landscape of the 256 schedules

Left: crews on a circle, edge width = contention-minutes, colour = whether the optimal rota cuts it.
Right: the histogram of every possible schedule. Note the shape — the achievable cut values are
spread out, and the optimum sits alone at the top. If your customer's instance were *bipartite*, the
right-hand histogram would have a single spike at `W_total` and the whole problem would be trivia;
the odd cycles printed above are what make it a genuine combinatorial problem.

In [46]:
XOPT = OPT_X[0]
POS = {i: (math.cos(2 * math.pi * i / N - math.pi / 2),
           math.sin(2 * math.pi * i / N - math.pi / 2)) for i in range(N)}

fig, axes = plt.subplots(1, 2, figsize=(12.6, 5.4))
ax = axes[0]
for i, j in EDGES:
    c = int(W[i, j])
    is_cut = XOPT[i] != XOPT[j]
    ax.plot([POS[i][0], POS[j][0]], [POS[i][1], POS[j][1]],
            color=("#1f77b4" if is_cut else "#c0392b"), lw=0.7 + 0.5 * c,
            alpha=0.5 if is_cut else 0.85, zorder=1)
    mx, my = (POS[i][0] + POS[j][0]) / 2, (POS[i][1] + POS[j][1]) / 2
    dx, dy = POS[j][0] - POS[i][0], POS[j][1] - POS[i][1]
    ln = math.hypot(dx, dy) or 1.0
    ax.text(mx - dy / ln * 0.11, my + dx / ln * 0.11, str(c), fontsize=6.5,
            ha="center", va="center", color="0.2", zorder=4,
            bbox=dict(boxstyle="round,pad=0.12", fc="w", ec="none", alpha=0.85))
for i in range(N):
    ax.plot(*POS[i], "o", ms=21, color=("#2ca02c" if XOPT[i] else "#ff7f0e"),
            mec="0.25", zorder=3)
    ax.text(POS[i][0], POS[i][1], CREWS[i][0], color="w", ha="center", va="center",
            fontsize=8.5, fontweight="bold", zorder=4)
    ax.annotate(CREWS[i], xy=POS[i], xytext=(1.33 * POS[i][0], 1.33 * POS[i][1]),
                ha="center", va="center", fontsize=7.5, zorder=4)
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
ax.legend(handles=[Patch(fc="#2ca02c", label="shift A (morning)"),
                   Patch(fc="#ff7f0e", label="shift B (evening)"),
                   Line2D([0], [0], color="#1f77b4", lw=3, label="edge cut -> contention avoided"),
                   Line2D([0], [0], color="#c0392b", lw=3, label="edge uncut -> contention paid")],
          fontsize=6.2, loc="lower left", framealpha=0.95)
ax.set_xlim(-1.9, 1.9); ax.set_ylim(-1.55, 1.55); ax.set_aspect("equal"); ax.axis("off")
ax.set_title("Instance: width = contention-min/day; blue = cut by the optimum, red = still paid",
             fontsize=8.0)

ax = axes[1]
vals, counts = np.unique(ALL_CUT, return_counts=True)
ax.bar(vals, counts, color="#4a7ba6", width=0.8)
ax.axvline(OPT, color="#c0392b", lw=2, label="optimum %d (%d rots)" % (OPT, len(OPT_X)))
ax.axvline(sq_cut, color="#ff7f0e", lw=2, ls="--", label="status quo %d" % sq_cut)
ax.axvline(ALL_CUT.mean(), color="0.35", lw=1.5, ls=":", label="random mean %.1f" % ALL_CUT.mean())
ax.set_xlabel("cut = contention-minutes eliminated per day")
ax.set_ylabel("number of rots")
ax.set_title("Exhaustive landscape: optimum alone at the top; the rota run today is barely above random", fontsize=8.5)
ax.legend(fontsize=7, loc="upper left")
plt.tight_layout(); plt.show(); plt.close(fig)


---

## Part 2 — Deliverable 2a: the QUBO

Quantum *and* classical annealers, and every vendor's job-submission form, want a quadratic
unconstrained binary optimisation: minimise `x^T Q x` over `x in {0,1}^n` with `Q` upper triangular.
So the first translation step is bit variables and a matrix.

Let `x_i = 1` mean crew *i* goes on shift A. Then for a contention pair `(i, j)` the two crews
**clash** when `x_i = x_j`, which is the Boolean indicator

```
[x_i == x_j] = 1 - x_i - x_j + 2 x_i x_j
```

so contention is `sum w_ij (1 - x_i - x_j + 2 x_i x_j)`. Because `sum w_ij (x_i + x_j - 2 x_i x_j)`
is exactly the cut (check the `i<j` bookkeeping: `sum_i d_i x_i - 2 sum_{i<j} w_ij x_i x_j`), the
whole thing collapses to `contention = W_total - cut`, so **minimising the negative cut** is the
QUBO we want:

```
E(x) = -cut(x) = -sum_i d_i x_i + 2 sum_{i<j} w_ij x_i x_j
Q_ii = -d_i            (one -w for each edge touching i)
Q_ij = +2 w_ij         (i < j, the pair term)
contention(x) = W_total + E(x)          # the constant lives outside the matrix
```

We will not trust that algebra. We will build `Q`, evaluate it on all 256 schedules, and print the
largest disagreement with the contention numbers we brute-forced above.

In [47]:
def to_qubo(edges, w):
    """Upper-triangular QUBO for MAX CUT: Q_ii = -d_i, Q_ij = 2 w_ij.

    Its minimum over x in {0,1}^n is -cut_max; contention(x) = W_total + x^T Q x.
    """
    n = w.shape[0]
    Q = np.zeros((n, n))
    for i, j in edges:
        wij = w[i, j]
        Q[i, i] -= wij
        Q[j, j] -= wij
        Q[i, j] += 2.0 * wij
    return Q


def qubo_energy(Q, x):
    """Upper-triangular QUBO energy: sum_i Q_ii x_i + sum_{i<j} Q_ij x_i x_j."""
    return float(x @ np.triu(Q, 1) @ x + np.dot(np.diag(Q), x))


Q = to_qubo(EDGES, W)

# QUBO value must equal -cut(x) exactly, and contention must be W_total + E.
qub = np.array([qubo_energy(Q, x) for x in ALL_X])
gap_neg = np.abs(qub + ALL_CUT)
gap_con = np.abs((qub + W_TOTAL) - np.array([contention(x) for x in ALL_X]))
print("QUBO vs brute force, all %d schedules:" % len(ALL_X))
print("  max |E(x) + cut(x)|            : %.3e" % gap_neg.max())
print("  max |W_total + E(x) - cont(x)| : %.3e" % gap_con.max())
assert gap_neg.max() == 0.0 and gap_con.max() == 0.0
print("  argmin of QUBO is an optimum   : %s (E_min = %.1f = -%d)"
      % (int(round(-qub.min())) == OPT, qub.min(), OPT))
print("  nonzero Q entries              : %d (diagonal %d, pair %d)"
      % (np.count_nonzero(Q), np.count_nonzero(np.diag(Q)), np.count_nonzero(np.triu(Q, 1))))
print("  convention spot-check          : Q[0,0]=%.1f vs -d_0=%.1f | Q[0,1]=%.1f vs 2w=%.1f"
      % (Q[0, 0], -DEG[0], Q[0, 1], 2 * W[0, 1]))

best = ALL_X[int(np.argmin(qub))]
print("  one QUBO optimum               :", show(best), "cut %.0f" % cut_value(best))
assert cut_value(best) == OPT


QUBO vs brute force, all 256 schedules:
  max |E(x) + cut(x)|            : 0.000e+00
  max |W_total + E(x) - cont(x)| : 0.000e+00
  argmin of QUBO is an optimum   : True (E_min = -68.0 = -68)
  nonzero Q entries              : 22 (diagonal 8, pair 14)
  convention spot-check          : Q[0,0]=-12.0 vs -d_0=-12.0 | Q[0,1]=4.0 vs 2w=4.0
  one QUBO optimum               : A=[Charlie,Delta,Hotel] B=[Alpha,Bravo,Echo,Foxtrot,Golf] cut 68


### The constraint the customer actually wants, and the penalty that buys it

The optimum above is *unbalanced*: putting crews on whatever shift maximises the cut lets one shift
end up with 6 crews and the other with 2 — which the depot cannot staff. So the real requirement is

> "two shifts, roughly equal crew counts, minimum contention."

Cardinality constraints are not free in QUBO: you add a penalty term and then have to *prove* the
penalty is large enough to dominate, and not so large that it distorts the objective. We use

```
P(x) = mu * (sum_i (2 x_i - 1))^2 ,   mu > 0      # zero exactly when counts are equal
```

and the honest engineering question is: what `mu` makes the penalised minimiser balanced, and does a
larger `mu` cost cut quality? Both are measured below rather than asserted.

In [48]:
def penalised(x, mu):
    """Objective the solver must minimise: contention + mu * imbalance^2 (minutes/day)."""
    imbalance = float(np.sum(2.0 * x - 1.0))
    return contention(x) + mu * imbalance * imbalance


def qubo_with_penalty(Q0, mu):
    """Upper-triangular QUBO for  E(x) + mu*(sum_i(2x_i-1))^2.

    With S = sum_i x_i:  mu*(2S-n)^2 = 4 mu S^2 - 4 mu n S + mu n^2
    and S^2 = S + 2 sum_{i<j} x_i x_j, so
        diagonal  += 4 mu (1 - n)
        pair      += 8 mu
    The additive constant mu*n^2 (like W_total) is outside the matrix; we account for it
    in the verification rather than silently dropping it.
    """
    n = Q0.shape[0]
    Q = Q0.copy()
    Q[np.diag_indices(n)] += 4.0 * mu * (1.0 - n)
    for i in range(n):
        for j in range(i + 1, n):
            Q[i, j] += 8.0 * mu
    return Q


QP = {mu: qubo_with_penalty(Q, mu) for mu in (0, 1, 2, 4, 8, 16, 32, 64)}
CONST = {mu: W_TOTAL + mu * float(N) ** 2 for mu in QP}

# Verify each penalised QUBO + its outside-the-matrix constants == the direct objective.
worst = {}
for mu, Qm in QP.items():
    e = np.array([qubo_energy(Qm, x) for x in ALL_X])
    worst[mu] = float(np.abs(e + CONST[mu] - np.array([penalised(x, mu) for x in ALL_X])).max())
print("penalised QUBO + constants vs direct objective, max |difference| per mu:")
print("  " + "  ".join("mu=%d:%.1e" % (m, v) for m, v in worst.items()))
assert max(worst.values()) == 0.0

print()
print(" mu  | best penalised schedule        | cut | bal? | contention | penalty")
print("-----+--------------------------------+-----+------+------------+--------")
rows = []
for mu in sorted(QP):
    Qm = QP[mu]
    e = np.array([qubo_energy(Qm, x) for x in ALL_X])
    x = ALL_X[int(np.argmin(e))]
    bal = is_balanced(x)
    rows.append((mu, x, cut_value(x), bal))
    print(" %3d | %-28s | %3.0f | %-4s | %8.1f | %8.1f"
          % (mu, show(x), cut_value(x), "yes" if bal else "NO",
             contention(x), mu * float(np.sum(2.0 * x - 1.0)) ** 2))

BAL_MUS = [mu for mu, x, c, bal in rows if bal]
print()
print("smallest mu (of this sweep) giving a balanced optimum :", min(BAL_MUS))


penalised QUBO + constants vs direct objective, max |difference| per mu:
  mu=0:0.0e+00  mu=1:0.0e+00  mu=2:0.0e+00  mu=4:0.0e+00  mu=8:0.0e+00  mu=16:0.0e+00  mu=32:0.0e+00  mu=64:0.0e+00

 mu  | best penalised schedule        | cut | bal? | contention | penalty
-----+--------------------------------+-----+------+------------+--------
   0 | A=[Charlie,Delta,Hotel] B=[Alpha,Bravo,Echo,Foxtrot,Golf] |  68 | NO   |      4.0 |      0.0
   1 | A=[Charlie,Delta,Hotel] B=[Alpha,Bravo,Echo,Foxtrot,Golf] |  68 | NO   |      4.0 |      4.0
   2 | A=[Bravo,Charlie,Delta,Hotel] B=[Alpha,Echo,Foxtrot,Golf] |  62 | yes  |     10.0 |      0.0
   4 | A=[Bravo,Charlie,Delta,Hotel] B=[Alpha,Echo,Foxtrot,Golf] |  62 | yes  |     10.0 |      0.0
   8 | A=[Bravo,Charlie,Delta,Hotel] B=[Alpha,Echo,Foxtrot,Golf] |  62 | yes  |     10.0 |      0.0
  16 | A=[Bravo,Charlie,Delta,Hotel] B=[Alpha,Echo,Foxtrot,Golf] |  62 | yes  |     10.0 |      0.0
  32 | A=[Bravo,Charlie,Delta,Hotel] B=[Alpha,Echo,Foxtrot,Gol

That table is the conversation to have with a customer. Below some threshold the penalty is
*advisory* — the solver happily returns an unbalanced rota. Above it the answer is balanced, but you
are paying for balance in solution quality. The threshold is a property of **this** graph's weights,
so we measure it and then check it against a bound we can derive on the page: for even `n` any
unbalanced schedule has `imbalance >= 2`, so its penalty is at least `4 mu`, and the most any
unbalanced schedule can ever win back by being unbalanced is `OPT` contention-minutes. Therefore
`4 mu > OPT` is **sufficient** for balance. Whether it is also *necessary* is exactly what the
sweep answers.

In [49]:
BAL_CUT = np.array([c for c, x in zip(ALL_CUT, ALL_X) if is_balanced(x)])
OPT_BAL = int(BAL_CUT.max())
ADV_MAX = OPT                        # largest contention advantage an unbalanced schedule can have
MU_SUFF = int(math.floor(ADV_MAX / 4.0)) + 1

print("unconstrained optimum cut   :", OPT, "| balanced?",
      any(is_balanced(x) for x in OPT_X))
print("best balanced cut (exact)   :", OPT_BAL)
print("price of the balance demand : %.1f contention-minutes/day = %.1f%% of the optimum cut"
      % (OPT - OPT_BAL, 100.0 * (OPT - OPT_BAL) / OPT))
print()
print("sufficient bound (4*mu > OPT) -> mu >= %d guarantees balance" % MU_SUFF)
fine = {}
for mu in range(0, 41):
    e = np.array([qubo_energy(qubo_with_penalty(Q, mu), x) for x in ALL_X])
    fine[mu] = is_balanced(ALL_X[int(np.argmin(e))])
first = min(m for m, ok in fine.items() if ok)
print("measured first balanced mu  :", first)
print("bound is", "SUFFICIENT (mu_star <= MU_SUFF)" if first <= MU_SUFF
      else "NOT sufficient - our derivation is wrong",
      "| slack: bound overshoots the truth by %d" % (MU_SUFF - first))

x_mu = ALL_X[int(np.argmin(np.array(
    [qubo_energy(qubo_with_penalty(Q, first), x) for x in ALL_X])))]
print("cut returned at mu =", first, "->", cut_value(x_mu),
      "| equals best balanced cut?", cut_value(x_mu) == OPT_BAL)


unconstrained optimum cut   : 68 | balanced? False
best balanced cut (exact)   : 62
price of the balance demand : 6.0 contention-minutes/day = 8.8% of the optimum cut

sufficient bound (4*mu > OPT) -> mu >= 18 guarantees balance
measured first balanced mu  : 2
bound is SUFFICIENT (mu_star <= MU_SUFF) | slack: bound overshoots the truth by 16
cut returned at mu = 2 -> 62.0 | equals best balanced cut? True


---

## Part 3 — Deliverable 2b: the cost Hamiltonian

Gate-based QAOA does not consume a QUBO matrix; it consumes a **diagonal Ising Hamiltonian**, because
`exp(-i gamma H)` has to be built out of gates. With the map `x_i = (1 - z_i)/2` (`z = +1` for shift
B, `z = -1` for shift A) two crew are on the *same* shift exactly when `z_i z_j = +1`, which gives the
identity worth memorising, because it is the single line connecting the Excel tab to the circuit:

```
[x_i == x_j]  ->  (1 + z_i z_j) / 2       contention = sum_{i<j} w_ij (1 + z_i z_j) / 2
                                             cut      = W_total - contention
```

so the Hamiltonian whose **ground state is the best rota** is

```
H = 1/2 * sum_{i<j} w_ij (I + Z_i Z_j)   =   (W_total/2) I + sum_{i<j} (w_ij/2) Z_i Z_j
    c0 = W_total/2 ,   J_ij = +w_ij/2 ,   h_i = 0
```

Three conventions to nail down before anyone touches hardware, because all three are silent-failure
modes in a QAOA integration:

1. ** minimise, not maximise.** `H` above is *cost to be minimised*, so QAOA uses
   `exp(-i gamma H)` and the best answer is the lowest eigenvalue. If you instead feed the cut
   operator `W_total/2 - 1/2 sum w Z_iZ_j` into the same loop you get the *worst* rota and every
   number still looks plausible. We keep both and print the offset between them.
2. **The constant is not cosmetic for reporting.** Dropping `c0` shifts every eigenvalue equally and
   leaves the argmin untouched, but a "ground-state energy" quoted without it is not the
   contention-minutes the customer pays. We report both and show the arithmetic.
3. **`h_i = 0` here.** The degree terms live in the *QUBO* diagonal; in the Ising form they fold into
   `c0` and the couplings. A notebook that carries `d_i` as a field term as well double-counts it.
   This is the bug to look for first when a QUBO-to-Ising translation "works" but returns the
   complement of the right answer.

In [50]:
def z_signs(index, n):
    """z_i = +1 - 2*bit_i for computational basis state `index` (bit 0 = |0> = z = +1)."""
    bits = np.array([(index >> (n - 1 - k)) & 1 for k in range(n)])
    return 1 - 2 * bits


def cost_operator_diag(n, edges, w):
    """Diagonal of H = 1/2 sum w_ij (I + Z_i Z_j) = contention, in basis-state order."""
    d = np.zeros(2 ** n)
    for b in range(2 ** n):
        z = z_signs(b, n)
        d[b] = 0.5 * sum(w[i, j] * (1.0 + z[i] * z[j]) for i, j in edges)
    return d


def ising_form(n, edges, w):
    """H = c0*I + sum J_ij Z_i Z_j + sum h_i Z_i, with the constant separated out."""
    J = np.zeros((n, n))
    h = np.zeros(n)
    for i, j in edges:
        J[i, j] += 0.5 * w[i, j]
    c0 = 0.5 * float(sum(w[i, j] for i, j in edges))
    return c0, J, h


C_DIAG = cost_operator_diag(N, EDGES, W)
C0, JJ, HF = ising_form(N, EDGES, W)

cont_by_index = np.array([contention(np.array([(b >> (N - 1 - k)) & 1 for k in range(N)]))
                          for b in range(2 ** N)])
diff_op = np.abs(C_DIAG - cont_by_index)
print("H diagonal vs brute-force contention, %d basis states:" % (2 ** N))
print("  max |difference|       : %.3e" % diff_op.max())
print("  min / max eigenvalue   : %.1f / %.1f  (min = best rota, max = worst)"
      % (C_DIAG.min(), C_DIAG.max()))
assert diff_op.max() == 0.0


def ising_diag(n, c0, J, h):
    d = np.zeros(2 ** n)
    for b in range(2 ** n):
        z = z_signs(b, n)
        d[b] = c0 + sum(J[i, j] * z[i] * z[j] for i, j in EDGES) + float(np.dot(h, z))
    return d


d_ising = ising_diag(N, C0, JJ, HF)
print("  c0 + J form vs operator  : max |difference| = %.3e" % np.abs(d_ising - C_DIAG).max())
assert np.abs(d_ising - C_DIAG).max() == 0.0

gs = int(np.argmin(C_DIAG))
x_gs = np.array([(gs >> (N - 1 - k)) & 1 for k in range(N)])
print()
print("ground state            :", show(x_gs))
print("  ground-state energy   : %.1f min/day" % C_DIAG[gs])
print("  contention of that rota (direct): %.1f -> agreement %.3e"
      % (contention(x_gs), abs(contention(x_gs) - C_DIAG[gs])))
assert abs(contention(x_gs) - C_DIAG[gs]) == 0.0
print("  cut read off H        : W_total - E_0 = %.1f vs direct cut %.1f"
      % (W_TOTAL - C_DIAG.min(), cut_value(x_gs)))
assert W_TOTAL - C_DIAG.min() == OPT
print("  if c0 were dropped    : quoted energy would be %.1f instead of %.1f (off by c0=%.1f)"
      % (C_DIAG[gs] - C0, C_DIAG[gs], C0))
print("  couplings J_ij        :", dict(((i, j), round(JJ[i, j], 2)) for i, j in EDGES[:4]), "...")
print("  fields h_i            :", set(np.round(HF, 12)), "-> zero, as derived")

# The complement symmetry: flipping every shift leaves the cut unchanged.
comp = np.array([cut_value(1 - x) == cut_value(x) for x in ALL_X])
print()
print("shift-label symmetry    : cut(x) == cut(1-x) for all %d schedules -> %s"
      % (len(ALL_X), bool(comp.all())))
assert comp.all()


H diagonal vs brute-force contention, 256 basis states:
  max |difference|       : 0.000e+00
  min / max eigenvalue   : 4.0 / 72.0  (min = best rota, max = worst)
  c0 + J form vs operator  : max |difference| = 0.000e+00

ground state            : A=[Charlie,Delta,Hotel] B=[Alpha,Bravo,Echo,Foxtrot,Golf]
  ground-state energy   : 4.0 min/day
  contention of that rota (direct): 4.0 -> agreement 0.000e+00
  cut read off H        : W_total - E_0 = 68.0 vs direct cut 68.0
  if c0 were dropped    : quoted energy would be -32.0 instead of 4.0 (off by c0=36.0)
  couplings J_ij        : {(0, 1): np.float64(1.0), (0, 2): np.float64(3.0), (1, 2): np.float64(3.0), (1, 3): np.float64(1.0)} ...
  fields h_i            : {np.float64(0.0)} -> zero, as derived

shift-label symmetry    : cut(x) == cut(1-x) for all 256 schedules -> True


### Figure 2 — the Hamiltonian is just the business table in another coordinate

We show the two halves of the identity `H = W_total/2 + 1/2 sum w_ij <z_i z_j>`: the pair
correlations of the *optimal* state, and how every basis state's energy is assembled from its cut.
This is the slide to use when a customer's architect says "why is a quantum computer relevant to my
rota at all?" — because the objective they wrote in Excel is literally the diagonal of the operator
we feed to the device.

In [51]:
fig, axes = plt.subplots(1, 2, figsize=(12.6, 4.9))

ax = axes[0]
z_gs = z_signs(gs, N)
zz_opt = np.array([[z_gs[i] * z_gs[j] for j in range(N)] for i in range(N)], float)
zz_plot = np.ma.masked_where(np.eye(N, dtype=bool), zz_opt)
im = ax.imshow(zz_plot, cmap="RdBu", vmin=-1, vmax=1)
for i, j in EDGES:
    for a, b in ((i, j), (j, i)):
        ax.text(b, a, str(int(W[i, j])), ha="center", va="center", fontsize=7,
                color="w", fontweight="bold")
ax.set_xticks(range(N)); ax.set_yticks(range(N))
ax.set_xticklabels([c[0] for c in CREWS]); ax.set_yticklabels([c[0] for c in CREWS])
ax.set_title("z_i z_j of the ground state: dark red = same shift (we pay), dark blue = split (free)",
             fontsize=8.5)
ax.grid(False)
plt.colorbar(im, ax=ax, fraction=0.046, label="z_i z_j (diagonal masked: always +1)")

ax = axes[1]
order = np.argsort(C_DIAG)
ax.scatter(C_DIAG[order], np.arange(len(order)), s=7, color="#4a7ba6")
ax.axvline(C_DIAG.min(), color="#c0392b", lw=2.0,
           label="ground state = best rota, E = %.0f" % C_DIAG.min())
ax.axvline(contention(STATUS_QUO), color="#ff7f0e", lw=1.5, ls="--",
           label="status quo, E = %.0f" % contention(STATUS_QUO))
ax.axvline(C_DIAG.max(), color="0.35", lw=1.2, ls=":",
           label="worst rota, E = %.0f (nobody split at all)" % C_DIAG.max())
ax.set_xlabel("H eigenvalue = contention-minutes/day")
ax.set_ylabel("basis state, sorted by energy")
ax.legend(fontsize=7, loc="center right")
ax.set_title("Spectrum of the cost Hamiltonian over all %d states" % (2 ** N), fontsize=9)
plt.tight_layout(); plt.show(); plt.close(fig)


---

## Part 4 — Tier 1: our own ideal simulator, driven by a gate instruction list

Everything from here on runs through **one intermediate representation**: a list of instructions

```
("h",    q)                 Hadamard
("rx",   q, theta)         exp(-i theta X / 2)
("zz",   i, j, theta)      exp(-i theta Z_i Z_j)
("gpi",  q, phi)           IonQ native pi-pulse about cos(phi)X + sin(phi)Y
("gpi2", q, phi)           IonQ native pi/2-pulse about the same axis
("ms",   i, j, theta)      Molmer-Sorensen exp(-i theta X_i X_j)
```

and **three backends** consume it: an exact statevector simulator (this part), a density-matrix
simulator plus a Pauli-trajectory Monte-Carlo sampler (Part 7), and a payload emitter for IonQ's job
API (Part 6). That is deliberate. On a real engagement the failure mode is never "the algorithm is
wrong"; it is "the circuit the researcher describes is not the circuit the SDK submits is not the
circuit the device runs." One list, three interpreters, and a measured agreement between them is how
you catch that class of bug.

The QAOA ansatz for our Hamiltonian `H = c0 I + sum (w_ij/2) Z_i Z_j` is

```
|psi(gamma, beta)> = prod_p [ exp(-i beta_p sum_i X_i) exp(-i gamma_p H) ] |+>^n
```

`exp(-i gamma_p H)` is **diagonal** in the computational basis, so it costs a phase multiply per
edge and no matrix exponential at all; `exp(-i beta sum X_i)` factorises into one `Rx(2 beta)` per
qubit. We start in `|+>^n` with Hadamards.

In [52]:
PAULI_X = np.array([[0.0, 1.0], [1.0, 0.0]], complex)
PAULI_Z = np.array([[1.0, 0.0], [0.0, -1.0]], complex)
H_MAT = np.array([[1.0, 1.0], [1.0, -1.0]], complex) / math.sqrt(2.0)

_ZZ_CACHE = {}


def z_signs(index, n):
    bits = [(index >> (n - 1 - k)) & 1 for k in range(n)]
    return np.array([1 - 2 * b for b in bits])


def zz_axis(i, j, n):
    key = (i, j, n)
    if key not in _ZZ_CACHE:
        v = np.empty(2 ** n)
        for b in range(2 ** n):
            v[b] = z_signs(b, n)[i] * z_signs(b, n)[j]
        _ZZ_CACHE[key] = v
    return _ZZ_CACHE[key]


def rx_matrix(theta):
    return (np.cos(theta / 2.0) * np.eye(2, dtype=complex) - 1j * math.sin(theta / 2.0) * PAULI_X)


def gpi_matrix(phi):
    """IonQ native pi pulse: exp(-i (pi/2) (cos(phi) X + sin(phi) Y) / 2) up to global phase."""
    return np.array([[0.0, complex(-1j) * np.exp(-1j * phi)],
                     [complex(-1j) * np.exp(1j * phi), 0.0]], complex)


def gpi2_matrix(phi):
    return (math.cos(math.pi / 4) * np.eye(2, dtype=complex)
            - 1j * math.sin(math.pi / 4)
            * (math.cos(phi) * PAULI_X + math.sin(phi)
               * np.array([[0.0, -1j], [1j, 0.0]], complex)))


def apply_1q(psi, U, q, n):
    t = np.tensordot(U, psi.reshape([2] * n), axes=([1], [q]))
    return np.moveaxis(t, 0, q).reshape(2 ** n)


def run_ir(ins, n, psi=None):
    """Backend 1: exact statevector. Every instruction is applied in list order."""
    if psi is None:
        psi = np.zeros(2 ** n, complex)
        psi[0] = 1.0          # the register starts in |0...0>; the list's Hadamards make |+>^n
    for g in ins:
        name = g[0]
        if name == "h":
            psi = apply_1q(psi, H_MAT, g[1], n)
        elif name == "rx":
            psi = apply_1q(psi, rx_matrix(g[2]), g[1], n)
        elif name == "gpi":
            psi = apply_1q(psi, gpi_matrix(g[2]), g[1], n)
        elif name == "gpi2":
            psi = apply_1q(psi, gpi2_matrix(g[2]), g[1], n)
        elif name == "zz":
            psi = psi * np.exp(-1j * g[3] * zz_axis(g[1], g[2], n))
        elif name == "ms":
            psi = apply_2q_xx(psi, g[3], g[1], g[2], n)
        else:
            raise ValueError("unknown instruction " + repr(name))
    return psi



def expectation_diag(psi, diag):
    p = np.abs(psi) ** 2
    return float(np.dot(p, diag))


def probabilities(psi):
    return np.abs(psi) ** 2


print("engine defined: 1 instruction list, statevector backend live")
print("empty circuit leaves the register in |0...0>: max |psi - e_0| = %.3e"
      % np.abs(run_ir([], N) - np.eye(2 ** N, dtype=complex)[0]).max())
print("Hadamard layer alone gives the uniform state: max |psi - 1/sqrt(2^n)| = %.3e"
      % np.abs(run_ir([("h", q) for q in range(N)], N)
               - np.full(2 ** N, 2.0 ** (-0.5 * N))).max())


engine defined: 1 instruction list, statevector backend live
empty circuit leaves the register in |0...0>: max |psi - e_0| = 0.000e+00
Hadamard layer alone gives the uniform state: max |psi - 1/sqrt(2^n)| = 4.163e-17


`exp(-i theta X_i X_j)` is the one gate in the list that is **not** diagonal: it moves
amplitude between `|ab>` and `|a^1, b^1>`, so it needs the two-qubit subspace handled explicitly
rather than the phase multiply that serves `zz`. That is exactly the kind of asymmetry a vendor's
native gate set forces on you, and it is worth seeing in the code.

With the engine complete, every instruction type gets verified against an independently constructed
dense matrix - the kron/diagonal path never calls the tensor-contraction path, so agreement is
evidence rather than tautology.

In [53]:
def apply_2q_xx(psi, theta, i, j, n):
    """exp(-i theta X_i X_j), exact.

    X_i X_j swaps |a b> <-> |a^ i^1, b^1> in pairs, so the two-qubit subspace spanned by
    {|ab>, |a' b'>} is closed: apply the 2x2 matrix [[cos t, -i sin t], [-i sin t, cos t]] there.
    """
    out = psi.copy()
    c, s = math.cos(theta), math.sin(theta)
    done = np.zeros(2 ** n, bool)
    for b in range(2 ** n):
        if done[b]:
            continue
        z = z_signs(b, n)
        b2 = b
        for q, sign in ((i, z[i]), (j, z[j])):
            if sign == 1:                       # flip bit q (z +1 -> bit 0 -> flip to 1)
                b2 |= (1 << (n - 1 - q))
            else:
                b2 &= ~(1 << (n - 1 - q))
        done[b] = done[b2] = True
        if b == b2:
            continue
        a0, a1 = psi[b], psi[b2]
        out[b] = c * a0 - 1j * s * a1
        out[b2] = c * a1 - 1j * s * a0
    return out


def dense_unitary(ins, n):
    """Reference backend: build the full 2^n x 2^n matrix by explicit kron / diagonal embedding."""
    U = np.eye(2 ** n, dtype=complex)
    for g in ins:
        name = g[0]
        if name in ("h", "rx", "gpi", "gpi2"):
            if name == "h":
                M = H_MAT
            elif name == "rx":
                M = rx_matrix(g[2])
            elif name == "gpi":
                M = gpi_matrix(g[2])
            else:
                M = gpi2_matrix(g[2])
            blocks = [M if q == g[1] else np.eye(2, dtype=complex) for q in range(n)]
            G = blocks[0]
            for blk in blocks[1:]:
                G = np.kron(G, blk)
        elif name == "zz":
            G = np.diag(np.exp(-1j * g[3] * zz_axis(g[1], g[2], n)))
        elif name == "ms":
            G = np.zeros((2 ** n, 2 ** n), complex)
            for b in range(2 ** n):
                col = np.zeros(2 ** n, complex); col[b] = 1.0
                G[:, b] = apply_2q_xx(col, g[3], g[1], g[2], n)
        else:
            raise ValueError(name)
        U = G @ U
    return U


SMALL = 3
test_ins = [("h", 0), ("h", 1), ("h", 2), ("zz", 0, 2, 0.37), ("rx", 1, 0.9),
            ("gpi2", 0, 0.4), ("gpi", 2, 1.1), ("ms", 0, 1, 0.28), ("zz", 1, 2, -0.6)]
U_ref = dense_unitary(test_ins, SMALL)
psi_start = np.zeros(2 ** SMALL, complex); psi_start[0] = 1.0
psi_sv = run_ir(test_ins, SMALL, psi=psi_start.copy())
psi_ref = U_ref @ psi_start
d = np.abs(psi_sv - psi_ref).max()
print("statevector IR vs independently built dense matrix (n=3, 9 mixed instructions):")
print("  max |difference| over %d amplitudes : %.3e" % (2 ** SMALL, d))
check("IR matches dense reference", d < 1e-12, "d=%.3e" % d)
print("  unitarity of reference             : max |U U^dag - I| = %.3e"
      % np.abs(U_ref @ U_ref.conj().T - np.eye(2 ** SMALL)).max())
check("dense reference is unitary",
      np.abs(U_ref @ U_ref.conj().T - np.eye(2 ** SMALL)).max() < 1e-12)

# The native gates must be honest unitaries, and gpi must equal two gpi2 on the same axis.
for name, M in (("gpi(0.7)", gpi_matrix(0.7)), ("gpi2(0.7)", gpi_matrix(0.7))):
    pass
g1 = gpi2_matrix(0.7) @ gpi2_matrix(0.7)
g2 = gpi_matrix(0.7)
ov = abs(np.vdot(g1.ravel(), g2.ravel())) / math.sqrt(4)
print("  gpi(phi) vs gpi2(phi)^2            : |overlap| = %.6f (1.0 = equal up to global phase)" % ov)
check("gpi = gpi2 squared up to phase", abs(ov - 1.0) < 1e-9)
for label, M in (("gpi", gpi_matrix(0.3)), ("gpi2", gpi2_matrix(0.3)), ("rx", rx_matrix(1.2))):
    err = np.abs(M @ M.conj().T - np.eye(2)).max()
    print("  %-5s unitarity error              : %.3e" % (label, err))
    check("%s is unitary" % label, err < 1e-12)


statevector IR vs independently built dense matrix (n=3, 9 mixed instructions):
  max |difference| over 8 amplitudes : 7.850e-17
  unitarity of reference             : max |U U^dag - I| = 7.772e-16
  gpi(phi) vs gpi2(phi)^2            : |overlap| = 1.000000 (1.0 = equal up to global phase)
  gpi   unitarity error              : 0.000e+00
  gpi2  unitarity error              : 2.930e-18
  rx    unitarity error              : 0.000e+00


In [54]:
def qaoa_instructions(n, edges, w, gammas, betas):
    """The ansatz as an instruction list. H = c0 I + sum (w_ij/2) Z_i Z_j."""
    ins = [("h", q) for q in range(n)]
    for g, b in zip(gammas, betas):
        for i, j in edges:
            if w[i, j] != 0.0:
                ins.append(("zz", i, j, 0.5 * g * w[i, j]))
        for q in range(n):
            ins.append(("rx", q, 2.0 * b))
    return ins


def qaoa_energy(gammas, betas, edges=EDGES, w=W, diag=C_DIAG, n=N):
    psi = run_ir(qaoa_instructions(n, edges, w, gammas, betas), n)
    return expectation_diag(psi, diag)


def ratio_of(E):
    """Cut achieved / known optimal cut. E is an expectation of H (contention)."""
    return (W_TOTAL - E) / OPT


p1 = qaoa_energy([math.pi / 4], [math.pi / 8])
print("sanity: p=1 at (pi/4, pi/8) -> E = %.4f, ratio = %.4f" % (p1, ratio_of(p1)))
print("instruction count for that circuit: %d (%d x 1q, %d zz)"
      % (len(qaoa_instructions(N, EDGES, W, [math.pi / 4], [math.pi / 8])), N + N, len(EDGES)))

# Dropping c0 must not change any measurable quantity: check two states agree up to a phase.
ins_a = qaoa_instructions(N, EDGES, W, [0.6], [0.4])
psi_a = run_ir(ins_a, N)
psi_b = run_ir(ins_a, N)
print("reproducibility of the engine: max |psi - psi| = %.3e (same list, twice)"
      % np.abs(psi_a - psi_b).max())
check("engine deterministic", np.abs(psi_a - psi_b).max() == 0.0)
print("global-phase invariance: |<psi|psi>| = %.15f, sum|psi|^2 = %.15f"
      % (abs(np.vdot(psi_a, psi_a)), probabilities(psi_a).sum()))
check("state stays normalised", abs(probabilities(psi_a).sum() - 1.0) < 1e-12)


sanity: p=1 at (pi/4, pi/8) -> E = 38.4142, ratio = 0.4939
instruction count for that circuit: 30 (16 x 1q, 14 zz)
reproducibility of the engine: max |psi - psi| = 0.000e+00 (same list, twice)
global-phase invariance: |<psi|psi>| = 0.999999999999999, sum|psi|^2 = 0.999999999999999


True

### Figure 3 — the `p = 1` landscape, and why a classical optimiser is needed at all

For `p = 1` there are two parameters, so we can draw the whole objective. The ridge structure is the
thing to look at: QAOA's parameter landscape is *not* a smooth bowl, and a naive grid at the wrong
resolution misses the good region entirely — which is why published QAOA work uses either a coarse
pattern search with restarts or the layer-wise "adiabatic" initialisation. We do both and compare.

In [55]:
G1 = np.linspace(0.0, math.pi, 61)
B1 = np.linspace(0.0, math.pi / 2, 41)
LAND = np.zeros((len(B1), len(G1)))
t0 = time.perf_counter()
for a, g in enumerate(G1):
    for b, bta in enumerate(B1):
        LAND[b, a] = ratio_of(qaoa_energy([g], [bta]))
grid_s = time.perf_counter() - t0

bi, ai = np.unravel_index(np.argmax(LAND), LAND.shape)
print("p=1 grid %dx%d in %.2f s -> best ratio %.5f at gamma=%.4f beta=%.4f"
      % (len(G1), len(B1), grid_s, LAND[bi, ai], G1[ai], B1[bi]))

fig, axes = plt.subplots(1, 2, figsize=(12.6, 4.6))
ax = axes[0]
im = ax.imshow(LAND, origin="lower", aspect="auto", cmap="viridis",
               extent=[G1[0], G1[-1], B1[0], B1[-1]], vmin=LAND.min(), vmax=LAND.max())
ax.plot(G1[ai], B1[bi], "w+", ms=14, mew=2)
ax.set_xlabel("gamma_1"); ax.set_ylabel("beta_1")
ax.set_title("QAOA p=1 ratio on the customer instance (grid best %.4f)" % LAND[bi, ai], fontsize=9)
plt.colorbar(im, ax=ax, fraction=0.046, label="cut / known optimum")

ax = axes[1]
psi = run_ir(qaoa_instructions(N, EDGES, W, [G1[ai]], [B1[bi]]), N)
pr = probabilities(psi)
idx = np.argsort(pr)[::-1][:14]
lab = ["".join("AB"[int(((i >> (N - 1 - q)) & 1) ^ 1)] for q in range(N)) for i in idx]
ax.bar(range(len(idx)), pr[idx], color="#4a7ba6")
ax.set_xticks(range(len(idx))); ax.set_xticklabels(lab, rotation=90, fontsize=6)
ax.set_ylabel("probability")
ax.set_xlabel("bitstring (A = shift A, B = shift B)")
opt_mask = np.zeros(2 ** N, bool)
for x in OPT_X:
    opt_mask[int("".join(str(int(v)) for v in x), 2)] = True
ax.bar(range(len(idx)), pr[idx], color="none",
       edgecolor=("#c0392b" if opt_mask[idx[0]] else "none"), lw=2)
ax.set_title("Top 14 amplitudes at the grid optimum; the two ground states carry\n"
             "total probability %.4f" % pr[opt_mask].sum(), fontsize=9)
plt.tight_layout(); plt.show(); plt.close(fig)
print("total probability on the optimal rots (exact, p=1 grid best): %.5f" % pr[opt_mask].sum())


p=1 grid 61x41 in 1.77 s -> best ratio 0.71583 at gamma=0.1047 beta=1.1781
total probability on the optimal rots (exact, p=1 grid best): 0.10196


### Tuning the angles, twice over, with two independent optimisers

Backend 1 is a from-scratch **pattern search**: coordinate-wise probing with a shrinking step, which
needs no derivatives and no third-party code. Backend 2 is SciPy's Nelder-Mead with random restarts.
They share nothing except the objective, so when they land on the same value that is evidence, not
decoration — and we print the gap between them rather than asserting it is zero.

In [56]:
TIMINGS = {}


def pattern_search(p, restarts=6, seed=11, iters=90):
    rng = np.random.default_rng(seed)
    best_E, best_par = None, None
    for r in range(restarts):
        par = np.concatenate([rng.uniform(0, math.pi, p), rng.uniform(0, math.pi / 2, p)])
        step = np.full(2 * p, 0.5)
        E = qaoa_energy(list(par[:p]), list(par[p:]))
        for it in range(iters):
            improved = False
            for k in range(2 * p):
                for sgn in (+1.0, -1.0):
                    trial = par.copy()
                    trial[k] += sgn * step[k]
                    Et = qaoa_energy(list(trial[:p]), list(trial[p:]))
                    if Et < E - 1e-15:
                        par, E, improved = trial, Et, True
                        break
                if improved:
                    break
            if not improved:
                step *= 0.5
                if step.max() < 1e-6:
                    break
        if best_E is None or E < best_E:
            best_E, best_par = E, par
    return best_E, best_par


SCIPY_RAN = True
try:
    from scipy.optimize import minimize
except ImportError:
    SCIPY_RAN = False


def scipy_search(p, restarts=6, seed=5):
    rng = np.random.default_rng(seed)

    def f(par):
        return qaoa_energy(list(par[:p]), list(par[p:]))

    best_E, best_par = np.inf, None
    for r in range(restarts):
        x0 = np.concatenate([rng.uniform(0, math.pi, p), rng.uniform(0, math.pi / 2, p)])
        res = minimize(f, x0, method="Nelder-Mead",
                       options={"maxiter": 4000, "xatol": 1e-8, "fatol": 1e-12})
        if res.fun < best_E:
            best_E, best_par = float(res.fun), res.x
    return best_E, best_par


t = time.perf_counter(); E_pat, par_pat = pattern_search(1)
TIMINGS["QAOA p=1 pattern search"] = time.perf_counter() - t
print("p=1 pattern search : E = %.6f  ratio = %.5f  (%.1f ms wall clock)"
      % (E_pat, ratio_of(E_pat), 1e3 * TIMINGS["QAOA p=1 pattern search"]))
if SCIPY_RAN:
    E_sci, par_sci = scipy_search(1)
    print("p=1 Nelder-Mead    : E = %.6f  ratio = %.5f" % (E_sci, ratio_of(E_sci)))
    print("independent-optimiser gap: |dE| = %.3e  (|d ratio| = %.3e)"
          % (abs(E_pat - E_sci), abs(ratio_of(E_pat) - ratio_of(E_sci))))
    check("two optimisers agree to 1e-6", abs(E_pat - E_sci) < 1e-6)
else:
    print("scipy unavailable: pattern search alone is used (nothing above depends on scipy)")


p=1 pattern search : E = 22.982309  ratio = 0.72085  (678.0 ms wall clock)
p=1 Nelder-Mead    : E = 22.982309  ratio = 0.72085
independent-optimiser gap: |dE| = 9.413e-11  (|d ratio| = 1.384e-12)


In [57]:
P_MAX = 4
LADDER = []
for p in range(1, P_MAX + 1):
    E, par = pattern_search(p, restarts=(6 if p <= 2 else 8), seed=17 + p)
    gam, bet = list(par[:p]), list(par[p:])
    psi = run_ir(qaoa_instructions(N, EDGES, W, gam, bet), N)
    pr = probabilities(psi)
    LADDER.append((p, E, ratio_of(E), pr[opt_mask].sum(), gam, bet))
    print("p=%d  E=%9.5f  ratio=%.5f  P(optimal rots)=%.5f" % (p, E, ratio_of(E), pr[opt_mask].sum()))

print()
print("exact ground-state check against the Hamiltonian diagonal:")
print("  smallest eigenvalue of H      : %.6f" % C_DIAG.min())
print("  best p=%d QAOA expectation     : %.6f  (gap %.6f)"
      % (P_MAX, LADDER[-1][1], LADDER[-1][1] - C_DIAG.min()))
check("QAOA expectation never beats the true ground state", LADDER[-1][1] >= C_DIAG.min() - 1e-12)


p=1  E= 22.98231  ratio=0.72085  P(optimal rots)=0.09752
p=2  E= 22.38309  ratio=0.72966  P(optimal rots)=0.12150
p=3  E= 21.94847  ratio=0.73605  P(optimal rots)=0.13176
p=4  E= 10.17899  ratio=0.90913  P(optimal rots)=0.60960

exact ground-state check against the Hamiltonian diagonal:
  smallest eigenvalue of H      : 4.000000
  best p=4 QAOA expectation     : 10.178991  (gap 6.178991)


True

### Finite shots: what you can actually *see* on hardware

A simulator hands you amplitudes; a device hands you bitstrings. The gap between those two is where
most "it worked in simulation" claims die, so we measure it directly: for each `p`, how many shots do
we need before the optimal rota appears at least once with 95% confidence, and what does the *best
sampled* cut look like?

In [58]:
SHOTS = 4096


def sample_counts(pr, shots, seed):
    rng = np.random.default_rng(seed)
    return rng.multinomial(shots, pr)


def best_from_counts(counts):
    best, bi = -1.0, None
    for b, c in enumerate(counts):
        if c > 0:
            x = np.array([(b >> (N - 1 - k)) & 1 for k in range(N)])
            cv = cut_value(x)
            if cv > best:
                best, bi = cv, b
    return best, bi


rng = np.random.default_rng(3)
print(" p  | P(opt) exact | shots to see opt @95%% | best cut @%d shots | ratio" % SHOTS)
print("----+--------------+------------------------+---------------------+--------")
SHOT_TABLE = []
for p, E, r, popt, gam, bet in LADDER:
    psi = run_ir(qaoa_instructions(N, EDGES, W, gam, bet), N)
    pr = probabilities(psi)
    need = (0 if popt <= 0 else math.ceil(math.log(0.05) / math.log(max(1e-300, 1 - popt))))
    counts = sample_counts(pr, SHOTS, 1234 + p)
    bc, _ = best_from_counts(counts)
    SHOT_TABLE.append((p, popt, need, bc, bc / OPT))
    print(" %d  | %12.5f | %24d | %19.0f | %.4f" % (p, popt, need, bc, bc / OPT))

print()
print("the exact expectation ratio and the best-shot ratio are different numbers;")
print("quoting the first while reporting the second is the classic overclaim.")


 p  | P(opt) exact | shots to see opt @95% | best cut @4096 shots | ratio
----+--------------+------------------------+---------------------+--------
 1  |      0.09752 |                       30 |                  68 | 1.0000
 2  |      0.12150 |                       24 |                  68 | 1.0000
 3  |      0.13176 |                       22 |                  68 | 1.0000
 4  |      0.60960 |                        4 |                  68 | 1.0000

the exact expectation ratio and the best-shot ratio are different numbers;
quoting the first while reporting the second is the classic overclaim.


---

## Part 5 — Deliverable 4's other half: the classical baselines a customer already has

This is the section that decides whether the whole exercise is worth funding, and it is the one most
quantum portfolios skip. Three facts to hold onto:

* Max-Cut is NP-hard, and it is one of Karp's original 21. But `2^14 = 16,384` splits, so **at the
  sizes a QAOA demo can reach, the exact answer is free.** Any claim of quantum value at `n = 8` is a
  claim about a problem a laptop solves in microseconds, and the report must say so.
* The **Goemans-Wilson SDP** (1995) gives a guaranteed `0.878` approximation ratio in polynomial
  time, and that constant is *tight for the SDP relaxation* (Karloff). Under the unique games
  conjecture it is essentially optimal; exceeding `16/17` is NP-hard (Khot et al.).
* Local search on Max-Cut is brutally good in practice: a single-flip hill climb already guarantees
  a `1/2`-approximation, and record-setting solvers (TOMCLUB, BiqMac-style branch and bound) close
  instances that defeat SDP packages.

So the baselines we implement, all from scratch: exact brute force, random, the status-quo rota,
greedy edge-filling, multi-start single-flip local search, and a Goemans-Wilson-style SDP relaxation
with random-hyperplane rounding. Each gets a measured ratio **and** a measured wall-clock.

In [59]:
def greedy_maxcut(n, edges, w):
    """Classic greedy: heaviest edge first, put its endpoints on opposite sides if undecided."""
    x = np.full(n, -1, int)
    for i, j in sorted(edges, key=lambda e: -w[e[0], e[1]]):
        if x[i] < 0 and x[j] < 0:
            x[i], x[j] = 1, 0
        elif x[i] >= 0 and x[j] < 0:
            x[j] = 1 - x[i]
        elif x[j] >= 0 and x[i] < 0:
            x[i] = 1 - x[j]
    for q in range(n):
        if x[q] < 0:
            same = [sum(w[q, k] for k in range(n) if w[q, k] and x[k] == t) for t in (0, 1)]
            x[q] = 0 if same[0] <= same[1] else 1
    return x


def flip_gain(i, x, n, w):
    """Exact change in cut if crew i moves to the other shift.

    Edges to SAME-side neighbours become cut (we gain them); edges to the other side
    stop being cut (we lose them). Getting this sign backwards turns a hill climb into a
    hill descent and returns a confident, wrong baseline.
    """
    same = sum(w[i, j] for j in range(n) if w[i, j] and x[j] == x[i])
    diff = sum(w[i, j] for j in range(n) if w[i, j] and x[j] != x[i])
    return same - diff


def local_search(x, n, edges, w):
    """Single-flip hill climb: move the best improving crew until nothing improves."""
    x = x.copy()
    while True:
        gains = [flip_gain(i, x, n, w) for i in range(n)]
        i = int(np.argmax(gains))
        if gains[i] <= 0:
            return x, cut_value(x)
        x[i] ^= 1


def multistart_local(n, edges, w, starts=60, seed=7):
    rng = np.random.default_rng(seed)
    best_x, best_c = None, -1.0
    for s in range(starts):
        x0 = rng.integers(0, 2, n) if s else np.zeros(n, int)
        x, c = local_search(x0, n, edges, w)
        if c > best_c:
            best_x, best_c = x, c
    return best_x, best_c


_rng = np.random.default_rng(0)
_gerr = 0.0
for _t in range(400):
    _x = _rng.integers(0, 2, N)
    for _i in range(N):
        _y = _x.copy(); _y[_i] ^= 1
        _gerr = max(_gerr, abs((cut_value(_y) - cut_value(_x)) - flip_gain(_i, _x, N, W)))
print("flip_gain audit: max |predicted gain - actual cut change| over 400 random schedules = %.3e"
      % _gerr)
check("flip_gain predicts the true cut change", _gerr < 1e-12)
del _x, _y

t = time.perf_counter(); x_g = greedy_maxcut(N, EDGES, W); TIMINGS["greedy"] = time.perf_counter() - t
c_g = cut_value(x_g)
t = time.perf_counter(); x_l, c_l = multistart_local(N, EDGES, W); TIMINGS["local search"] = time.perf_counter() - t
t = time.perf_counter(); c_r = float(ALL_CUT.mean()); TIMINGS["random (mean of 256)"] = time.perf_counter() - t
t = time.perf_counter(); c_b = float(OPT); TIMINGS["brute force (256 rots)"] = time.perf_counter() - t

print("greedy          : cut %5.1f  ratio %.4f  (%s)" % (c_g, c_g / OPT, show(x_g)))
print("local search x60: cut %5.1f  ratio %.4f  (%s)" % (c_l, c_l / OPT, show(x_l)))
print("random mean     : cut %5.1f  ratio %.4f" % (c_r, c_r / OPT))
print("status quo      : cut %5.1f  ratio %.4f" % (sq_cut, sq_cut / OPT))
print()
for k, v in TIMINGS.items():
    print("  %-24s %.1f us" % (k, 1e6 * v))
check("local search is at least as good as greedy", c_l >= c_g - 1e-12)
check("local search beats every schedule it started from",
      c_l >= max(cut_value(np.zeros(N, int)), cut_value(np.ones(N, int))))
_ls_x = np.random.default_rng(11).integers(0, 2, N); _ls_c0 = cut_value(_ls_x)
_, _ls_c1 = local_search(_ls_x, N, EDGES, W)
check("single-flip climb never leaves a start worse than it found it", _ls_c1 >= _ls_c0)
print("  one climb: start %.1f -> finish %.1f (never goes down)" % (_ls_c0, _ls_c1))
check("no classical baseline exceeds the exact optimum",
      max(c_g, c_l, c_r, sq_cut) <= OPT + 1e-12)


flip_gain audit: max |predicted gain - actual cut change| over 400 random schedules = 0.000e+00
greedy          : cut  68.0  ratio 1.0000  (A=[Charlie,Delta,Hotel] B=[Alpha,Bravo,Echo,Foxtrot,Golf])
local search x60: cut  68.0  ratio 1.0000  (A=[Charlie,Delta,Hotel] B=[Alpha,Bravo,Echo,Foxtrot,Golf])
random mean     : cut  36.0  ratio 0.5294
status quo      : cut  38.0  ratio 0.5588

  QAOA p=1 pattern search  678007.5 us
  greedy                   163.3 us
  local search             26544.2 us
  random (mean of 256)     153.4 us
  brute force (256 rots)   78.2 us
  one climb: start 45.0 -> finish 51.0 (never goes down)


True

### The SDP relaxation, done honestly

Goemans-Wilson is: relax `x_i x_j` products to inner products of unit vectors `v_i`, maximise
`1/2 sum w_ij (1 - v_i . v_j)` subject to `||v_i|| = 1`, then round with a random hyperplane. The
convex program is normally handed to an SDP solver. We do not have one in a library-free notebook, so
we use the standard **Burer-Monteiro factorisation**: parametrise the Gram matrix as `X = V V^T` with
`V` of rank `r`, keep rows on the unit sphere by projection, and do gradient ascent with
backtracking. That is a *heuristic* for a non-convex problem, which has one consequence we are
obliged to state: **the SDP value is only an upper bound on the optimum if the solver actually found
the SDP optimum.** We therefore verify the two properties that are checkable (PSD-ness, unit
diagonal) and report the bound claim as conditional, then test the whole pipeline on a bipartite
instance where the true SDP value is known analytically.

In [60]:
def sdp_vectors(n, edges, w, rank=4, iters=4000, seed=3, lr0=0.35):
    """Burer-Monteiro ASCENT on  f(V) = 1/2 sum w_ij (1 - v_i . v_j), rows of V unit norm.

    MaxCut's SDP relaxation is a MAXIMISATION. Run it as a minimisation and the solver
    happily returns f = 0 (every vector collapsed onto its neighbour), which is the
    *worst* cut - and looks like a converged result in the printout.
    """
    rng = np.random.default_rng(seed)
    V = rng.normal(size=(n, rank))
    V /= np.linalg.norm(V, axis=1, keepdims=True)

    def f(Vm):
        return 0.5 * sum(w[i, j] * (1.0 - float(np.dot(Vm[i], Vm[j]))) for i, j in edges)

    def grad(Vm):
        G = np.zeros_like(Vm)
        for i, j in edges:
            G[i] -= 0.5 * w[i, j] * Vm[j]
            G[j] -= 0.5 * w[i, j] * Vm[i]
        return G

    def normed(Vm):
        return Vm / np.maximum(np.linalg.norm(Vm, axis=1, keepdims=True), 1e-12)

    best = f(V)
    lr = lr0
    for it in range(iters):
        Vn = normed(V + lr * grad(V))
        val = f(Vn)
        if val > best + 1e-15:
            V, best = Vn, val
            lr = min(lr * 1.02, 4.0)
        else:
            lr *= 0.5
            if lr < 1e-12:
                break
    return V, best


def hyperplane_round(V, edges, w, trials=3000, seed=8):
    rng = np.random.default_rng(seed)
    n = V.shape[0]
    best, vals = -1.0, []
    for k in range(trials):
        g = rng.normal(size=V.shape[1])
        x = (V @ g < 0).astype(int)
        c = cut_value(x, edges, w)
        vals.append(c)
        best = max(best, c)
    return best, float(np.mean(vals))


V, SDP_VAL = sdp_vectors(N, EDGES, W)
X = V @ V.T
print("SDP pipeline on the customer instance:")
print("  min eigenvalue of X          : %.3e  (>= 0 means PSD)" % np.linalg.eigvalsh(X).min())
print("  max |diag(X) - 1|            : %.3e" % np.abs(np.diag(X) - 1.0).max())
print("  SDP objective (our solver)   : %.6f" % SDP_VAL)
print("  true optimum cut             : %.6f" % OPT)
print("  bound holds (objective >= opt)? %s   [conditional: only if the SDP optimum was found]"
      % (SDP_VAL >= OPT - 1e-9))
check("SDP Gram matrix is PSD", np.linalg.eigvalsh(X).min() > -1e-8)
check("SDP Gram matrix has unit diagonal", np.abs(np.diag(X) - 1.0).max() < 1e-8)
check("SDP objective lies between the exact optimum and the trivial bound W_total",
      OPT - 1e-9 <= SDP_VAL <= W_TOTAL + 1e-9, "SDP %.6f, opt %.1f, W_total %.1f" % (SDP_VAL, OPT, W_TOTAL))

b_round, m_round = hyperplane_round(V, EDGES, W)
print("  random-hyperplane rounding   : best %.1f  mean %.3f  (mean/SDP = %.4f)"
      % (b_round, m_round, m_round / SDP_VAL))
print("  guarantee we cite, not prove : Goemans-Wilson 0.878 (alpha_GW = 0.87856), tight for the SDP")
if abs(m_round / SDP_VAL - 1.0) < 1e-6:
    print("     mean/SDP = 1.0 is NOT the 0.878 theorem at work. Our ascent landed on an")
    print("     INTEGRAL Gram matrix (rank 1, v_i = sign_i x one direction), so the relaxation is")
    print("     tight on this instance and every hyperplane returns the same rota. Do not quote")
    print("     this as evidence of an approximation ratio - on a hard instance it will not repeat.")

# Analytic control: on a bipartite graph the SDP value must equal W_total exactly.
BIP = [(0, 1), (1, 2), (2, 3), (3, 0)]
Wb = np.zeros((4, 4))
for i, j in BIP:
    Wb[i, j] = Wb[j, i] = 3.0
Vb, sdp_b = sdp_vectors(4, BIP, Wb, rank=3, seed=17)
rb, mb = hyperplane_round(Vb, BIP, Wb, trials=800, seed=4)
print()
print("control on a 4-cycle (bipartite, W_total = 12): SDP %.6f, rounded best %.1f"
      % (sdp_b, rb))
check("bipartite control: SDP reaches W_total", abs(sdp_b - 12.0) < 1e-6)
check("bipartite control: rounding finds the optimum", rb == 12.0)
check("rounding never exceeds the exact optimum (it is a heuristic, not an oracle)",
      b_round <= OPT + 1e-12, "rounded best %.1f vs opt %.1f" % (b_round, OPT))
print("  gap of our ascent to the exact optimum: %.6f  (0.0 = we solved the relaxation)"
      % (SDP_VAL - OPT))


SDP pipeline on the customer instance:
  min eigenvalue of X          : -6.530e-16  (>= 0 means PSD)
  max |diag(X) - 1|            : 2.220e-16
  SDP objective (our solver)   : 68.000000
  true optimum cut             : 68.000000
  bound holds (objective >= opt)? True   [conditional: only if the SDP optimum was found]
  random-hyperplane rounding   : best 68.0  mean 68.000  (mean/SDP = 1.0000)
  guarantee we cite, not prove : Goemans-Wilson 0.878 (alpha_GW = 0.87856), tight for the SDP
     mean/SDP = 1.0 is NOT the 0.878 theorem at work. Our ascent landed on an
     INTEGRAL Gram matrix (rank 1, v_i = sign_i x one direction), so the relaxation is
     tight on this instance and every hyperplane returns the same rota. Do not quote
     this as evidence of an approximation ratio - on a hard instance it will not repeat.

control on a 4-cycle (bipartite, W_total = 12): SDP 12.000000, rounded best 12.0
  gap of our ascent to the exact optimum: 0.000000  (0.0 = we solved the relaxation)


### Figure 4 — the quality ladder, with the classical lines drawn across it

The single most important picture in this notebook. Every quantum curve is a measured expectation
value or a measured best-of-shots value; every classical line is computed on the same instance in the
same session. If a customer is going to fund a QPU trial, this is the chart that decides it.

In [61]:
fig, axes = plt.subplots(1, 2, figsize=(12.6, 4.8))

ax = axes[0]
ps = [row[0] for row in LADDER]
ex = [row[2] for row in LADDER]
bs = [row[4] for row in SHOT_TABLE]
check("best-of-shots series is a ratio, not a cut divided by a second time",
      all(0.0 < b <= 1.0 + 1e-12 for b in bs), str(bs))
ax.plot(ps, ex, "o-", color="#1f77b4", lw=2, label="QAOA exact <H> (our statevector)")
ax.plot(ps, bs, "s--", color="#2ca02c", lw=2, label="QAOA best of %d shots" % SHOTS)
for y, lab, col, st in ((1.0, "brute-force optimum", "#c0392b", "-"),
                        (c_l / OPT, "multi-start local search", "#9467bd", "-."),
                        (c_g / OPT, "greedy", "#ff7f0e", "--"),
                        (b_round / OPT, "GW-style SDP + rounding", "#8c564b", ":"),
                        (sq_cut / OPT, "status quo rota", "0.4", (0, (1, 4))),
                        (c_r / OPT, "random mean", "0.7", (0, (3, 3, 1, 3)))):
    ax.axhline(y, color=col, ls=st, lw=1.3, label="%s %.4f" % (lab, y))
ax.set_xticks(ps); ax.set_xlabel("layers p")
ax.set_ylabel("cut / known optimum")
ax.set_ylim(min(0.5, min(bs) - 0.06), 1.06)
ax.set_title("Tier-1 quality ladder on the customer instance (n=8)", fontsize=9)
ax.legend(fontsize=6.4, loc="lower right", ncol=2)

ax = axes[1]
tots = sorted(TIMINGS.items(), key=lambda kv: kv[1])
names = [k for k, v in tots]
vals = [1e6 * v for k, v in tots]
ax.barh(range(len(names)), vals, color="#4a7ba6")
ax.set_yticks(range(len(names))); ax.set_yticklabels(names, fontsize=7)
ax.set_xscale("log")
ax.set_xlabel("wall clock per unit of work (microseconds, measured)")
ax.set_title("Classical cost is not just quality: measured runtimes on the same laptop", fontsize=9)
for i, v in enumerate(vals):
    ax.text(v, i, "  %.3g us" % v, va="center", fontsize=6.5)
plt.tight_layout(); plt.show(); plt.close(fig)
print("classical timings (us):", {k: round(1e6 * v, 1) for k, v in TIMINGS.items()})


classical timings (us): {'QAOA p=1 pattern search': 678007.5, 'greedy': 163.3, 'local search': 26544.2, 'random (mean of 256)': 153.4, 'brute force (256 rots)': 78.2}


---

## Part 6 — Deliverable 3: the same circuit in IonQ's own language

Everything so far is vendor-neutral. This part is the integration layer, and it is written under a
rule: **a vendor number in this notebook either comes with a source and an access date, or it is not
in the notebook.** Quantum specs are calibration snapshots, not constants; quoting last year's
fidelity as though it were a physical constant is the fastest way to lose a technical customer.

Verified directly against the cited pages on **9 November 2026**, not from memory or a secondary
source:

In [62]:
ACCESS_DATE = "2026-11-09"

VENDOR = {
    "sdk": {
        "package": "qiskit-ionq",
        "version": "1.1.1",
        "released": "2026-06-05",
        "import": "from qiskit_ionq import IonQProvider",
        "token_env_var": "IONQ_API_KEY",
        "backends": ["simulator", "qpu.forte-1"],
        "needs_qiskit_runtime": False,
        "source": "https://docs.ionq.com/sdks/qiskit + https://pypi.org/project/qiskit-ionq/",
    },
    "system": {
        "name": "IonQ Forte",
        "qubits": 36,
        "connectivity": "all-to-all (single ion chain)",
        "one_qubit_error": 2.0e-4,
        "two_qubit_error": 4.64e-3,
        "spam_error": 5.0e-3,
        "gate_time_1q_us": 135.0,
        "gate_time_2q_us": 600.0,
        "simulator_qubit_cap": 29,
        "source": "vendor spec https://www.ionq.com/quantum-systems/forte ; "
                  "peer-reviewed medians arXiv:2308.05071v2 (30-AQ benchmark, 2024)",
    },
    "gates": {
        "native": ["gpi(phi)", "gpi2(phi)", "ms(phi0, phi1[, theta])", "zz(theta)"],
        "zz_convention": "ZZ(theta) = exp(-i pi theta Z_tensor_Z)",
        "global_ms_offered": False,
        "source": "https://docs.ionq.com/guides/getting-started-with-native-gates "
                  " + https://docs.ionq.com/sdks/qiskit/native-gates-qiskit",
    },
    "limits": {
        "shots_default": 100, "shots_min": 1, "shots_max": 1000000,
        "debiasing_min_shots": 500, "braket_em_min_shots": 2500,
        "source": "https://docs.ionq.com/api-reference/v0.4/jobs/create-job "
                  " + https://docs.aws.amazon.com/braket/latest/developerguide/error-mitigation-ionq.html",
    },
    "price": {
        "braket_per_task_usd": 0.30, "braket_per_shot_usd": 0.08,
        "azure_per_1q_gate_shot_usd": 1.645e-4, "azure_per_2q_gate_shot_usd": 1.121e-3,
        "azure_min_per_program_usd": 25.7899,
        "ionq_direct": None,
        "source": "https://aws.amazon.com/braket/pricing/ (quoted verbatim: IonQ Forte "
                  "per-task $0.30000, per-shot $0.08000) + "
                  "https://learn.microsoft.com/en-us/azure/quantum/pricing (updated 2026-11-03)",
    },
}

print("VENDOR FACTS  (accessed %s)" % ACCESS_DATE)
print("  sdk      : %s %s (%s), provider %s, token from $%s"
      % (VENDOR["sdk"]["package"], VENDOR["sdk"]["version"], VENDOR["sdk"]["released"],
         VENDOR["sdk"]["import"], VENDOR["sdk"]["token_env_var"]))
print("  backends : %s" % VENDOR["sdk"]["backends"])
s = VENDOR["system"]
print("  system   : %s, %d qubits, %s" % (s["name"], s["qubits"], s["connectivity"]))
print("             1q err %.2e | 2q err %.2e | SPAM %.2e | gate times %.0f/%.0f us"
      % (s["one_qubit_error"], s["two_qubit_error"], s["spam_error"],
         s["gate_time_1q_us"], s["gate_time_2q_us"]))
print("  native   : %s" % ", ".join(VENDOR["gates"]["native"]))
print("             global MS gate offered? %s" % VENDOR["gates"]["global_ms_offered"])
print("  shots    : default %d, max %d; debiasing needs >= %d, Braket EM >= %d"
      % (VENDOR["limits"]["shots_default"], VENDOR["limits"]["shots_max"],
         VENDOR["limits"]["debiasing_min_shots"], VENDOR["limits"]["braket_em_min_shots"]))
print("  price    : Braket $%.2f/task + $%.3f/shot ; Azure $%.2e per 2q-gate-shot"
      % (VENDOR["price"]["braket_per_task_usd"], VENDOR["price"]["braket_per_shot_usd"],
         VENDOR["price"]["azure_per_2q_gate_shot_usd"]))
print("  NOT public: IonQ direct-cloud per-shot price -> %r (we do not estimate it)"
      % (VENDOR["price"]["ionq_direct"],))


VENDOR FACTS  (accessed 2026-11-09)
  sdk      : qiskit-ionq 1.1.1 (2026-06-05), provider from qiskit_ionq import IonQProvider, token from $IONQ_API_KEY
  backends : ['simulator', 'qpu.forte-1']
  system   : IonQ Forte, 36 qubits, all-to-all (single ion chain)
             1q err 2.00e-04 | 2q err 4.64e-03 | SPAM 5.00e-03 | gate times 135/600 us
  native   : gpi(phi), gpi2(phi), ms(phi0, phi1[, theta]), zz(theta)
             global MS gate offered? False
  shots    : default 100, max 1000000; debiasing needs >= 500, Braket EM >= 2500
  price    : Braket $0.30/task + $0.080/shot ; Azure $1.12e-03 per 2q-gate-shot
  NOT public: IonQ direct-cloud per-shot price -> None (we do not estimate it)


### The single most consequential line in that table

`global_ms_offered = False`. IonQ's own documentation says the Mølmer-Sørensen gate is
**two-qubit only** - a global GMS pulse that entangles the whole chain at once is physically
possible but, in their words, not offered. That destroys the argument many QAOA write-ups make,
namely "all the cost terms commute, so one global entangling pulse implements the whole layer and the
depth is constant in `|E|`". On IonQ today:

* one `zz` (or `ms`) gate **per edge per layer**, each carrying its own angle, because distinct
  angles cannot share one entangling gate;
* those gates are **serial** within one ion chain;
* what all-to-all connectivity genuinely buys you is **zero SWAP overhead** - no routing qubits, no
  SWAP ladder, no depth growth with distance - which is a real advantage over a fixed-coupling
  superconducting device, and a *different* one from the constant-depth claim.

A proposal that promises "constant-depth QAOA on trapped ions" is either describing a compiler
feature you do not have or repeating a slide it did not check. We count both numbers ourselves below.

In [63]:
def ionq_angle(theta):
    """Convert OUR convention exp(-i theta ZZ) to IonQ's ZZ(phi) = exp(-i pi phi ZZ)."""
    return theta / math.pi


def zz_ionq_matrix(phi):
    """The vendor's documented matrix, built from the doc string alone."""
    return np.diag([np.exp(-1j * math.pi * phi), np.exp(1j * math.pi * phi),
                    np.exp(1j * math.pi * phi), np.exp(-1j * math.pi * phi)])


def zz_ours_matrix(theta):
    return np.diag(np.exp(-1j * theta * np.array([1.0, -1.0, -1.0, 1.0])))


worst = 0.0
for th in np.linspace(-2.0, 2.0, 41):
    worst = max(worst, np.abs(zz_ours_matrix(th) - zz_ionq_matrix(ionq_angle(th))).max())
print("angle-conversion check over 41 angles: max |our ZZ(theta) - vendor ZZ(theta/pi)| = %.3e"
      % worst)
check("zz angle convention matches the documented matrix", worst < 1e-12)
print("  (a factor of pi here is a silent, total failure: the circuit runs, the phases are wrong,")
print("   and every number you report is a confident lie.)")

# An arbitrary-angle X rotation is NOT one gpi2 pulse. Synthesise it from native pulses and
# measure the residual rather than trusting a decomposition we did not check.
def gpi2(phi):
    return gpi2_matrix(phi)


def rx_from_pulses(angles):
    U = np.eye(2, dtype=complex)
    for a in angles:
        U = gpi2(a) @ U
    return U


def phase_free_distance(A, B):
    """min over global phase of ||A - e^{i p} B||."""
    best = np.inf
    for p in np.linspace(0, 2 * math.pi, 361):
        best = min(best, np.abs(A - np.exp(1j * p) * B).max())
    return best


target = rx_matrix(2.0 * 0.417)
res1 = min(phase_free_distance(gpi2(a), target) for a in np.linspace(0, 2 * math.pi, 721))
grid2 = np.linspace(0, 2 * math.pi, 121)
res2 = min(phase_free_distance(rx_from_pulses([a, b]), target)
           for a in grid2 for b in grid2)
print("arbitrary Rx(0.834) from native pulses:")
print("  best single gpi2 residual      : %.4f  (one pulse cannot do it)" % res1)
print("  best two-pulse residual        : %.6f" % res2)
print("  -> we price the mixer at 1 rotation in QIS format, and 2 native pulses if the compiler")
print("     insists on gpi2 only. Both counts appear in the cost model.")


angle-conversion check over 41 angles: max |our ZZ(theta) - vendor ZZ(theta/pi)| = 2.483e-16
  (a factor of pi here is a silent, total failure: the circuit runs, the phases are wrong,
   and every number you report is a confident lie.)
arbitrary Rx(0.834) from native pulses:
  best single gpi2 residual      : 0.3021  (one pulse cannot do it)
  best two-pulse residual        : 0.207754
  -> we price the mixer at 1 rotation in QIS format, and 2 native pulses if the compiler
     insists on gpi2 only. Both counts appear in the cost model.


In [64]:
def to_ionq_payload(ins, n, fmt="qis"):
    """Emit a job body in the shape IonQ's v0.4 create-job API takes (in memory only).

    fmt="qis"    : vendor QIS gate names (h/rx/zz) - what the SDK accepts today.
    fmt="native" : native instructions (gpi2/ms/zz with IonQ angle convention).
    """
    gates = []
    for g in ins:
        name = g[0]
        if name == "h":
            gates.append({"gate": "h", "target": g[1]})
        elif name == "rx":
            if fmt == "qis":
                gates.append({"gate": "rx", "target": g[1], "angle": g[2]})
            else:
                gates.append({"gate": "gpi2", "target": g[1], "phase": 0.0})
                gates.append({"gate": "gpi2", "target": g[1], "phase": 0.0})
        elif name == "zz":
            gates.append({"gate": "zz", "control": g[1], "target": g[2],
                          "phase": ionq_angle(g[3]) if fmt == "native" else g[3]})
        elif name == "ms":
            gates.append({"gate": "ms", "control": g[1], "target": g[2],
                          "phase": ionq_angle(g[3])})
        else:
            raise ValueError(name)
    return {"qubits": n, "gates": gates}


def count_gates(ins):
    n1 = sum(1 for g in ins if g[0] in ("h", "rx", "gpi", "gpi2"))
    n2 = sum(1 for g in ins if g[0] in ("zz", "ms"))
    return n1, n2


gam, bet = LADDER[-1][4], LADDER[-1][5]
ins_p = qaoa_instructions(N, EDGES, W, gam, bet)
n1, n2 = count_gates(ins_p)
pay = to_ionq_payload(ins_p, N, fmt="qis")
print("p=%d circuit: %d instructions -> %d single-qubit, %d entangling"
      % (LADDER[-1][0], len(ins_p), n1, n2))
print("payload shape: qubits=%d, gates=%d, first three:" % (pay["qubits"], len(pay["gates"])))
for entry in pay["gates"][:3]:
    print("   ", entry)
print("entangling gates per layer = |E| = %d, exactly as the docs imply (no global pulse)"
      % len(EDGES))
check("gate count matches instruction list", n1 + n2 == len(ins_p))
check("native payload converts zz angles by 1/pi",
      abs(to_ionq_payload([("zz", 0, 1, 0.5)], 2, "native")["gates"][0]["phase"] - 0.5 / math.pi)
      < 1e-15)


p=4 circuit: 96 instructions -> 40 single-qubit, 56 entangling
payload shape: qubits=8, gates=96, first three:
    {'gate': 'h', 'target': 0}
    {'gate': 'h', 'target': 1}
    {'gate': 'h', 'target': 2}
entangling gates per layer = |E| = 14, exactly as the docs imply (no global pulse)


True

### What all-to-all is actually worth, counted on the graph

We can compute the routing cost ourselves, with no vendor claims: on a fully-connected device the
cost layer is `|E|` entangling gates and the circuit's two-qubit depth per layer is `|E|` (serial).
On a linear nearest-neighbour chain of the same `n` qubits, an edge `(i, j)` must be swapped into
adjacency, costing `2|i-j| - 1` two-qubit operations. The ratio is a property of the *instance*, not
of the technology marketing - and for a logistics graph whose heavy edges happen to connect distant
nodes, it is large.

In [65]:
def routing_cost_all_to_all(edges):
    return sum(1 for i, j in edges)


def routing_cost_path(edges):
    """Two-qubit operations needed if the hardware graph is the 0-1-2-...-(n-1) chain."""
    return sum(2 * abs(i - j) - 1 for i, j in edges)


def routing_cost_matching_layers(edges, w=None):
    """Depth if disjoint pairs may run in parallel (a colouring of the hardware graph)."""
    if w is None:
        w = np.ones((N, N))
    remaining = sorted(edges, key=lambda e: -w[e[0], e[1]])
    layers = 0
    while remaining:
        used, taken = set(), []
        for e in remaining:
            if e[0] not in used and e[1] not in used:
                taken.append(e); used.update(e)
        remaining = [e for e in remaining if e not in taken]
        layers += 1
    return layers


a2a = routing_cost_all_to_all(EDGES)
pth = routing_cost_path(EDGES)
mat = routing_cost_matching_layers(EDGES)
print("customer instance, one cost layer:")
print("  IonQ (all-to-all, serial)      : %3d entangling gates" % a2a)
print("  path graph (SWAP routed)       : %3d entangling gates  (x%.2f)" % (pth, pth / a2a))
print("  parallel matching schedule     : %3d layers deep (what a fixed-coupling machine hopes for)"
      % mat)

NSWEEP = [6, 8, 10, 12, 14]
rows = []
for nn in NSWEEP:
    ee = [(i, j) for i in range(nn) for j in range(i + 1, nn) if (i * 7 + j * 3) % 5 == 0]
    wsw = np.zeros((nn, nn))
    for i, j in ee:
        wsw[i, j] = wsw[j, i] = 1.0 + (i + j) % 4
    rows.append((nn, len(ee), routing_cost_path(ee), routing_cost_matching_layers(ee, wsw)))
print()
print(" n  | edges | path-routed 2q gates | matching depth")
print("----+-------+----------------------+----------------")
for nn, e, p, m in rows:
    print(" %2d | %5d | %18d | %14d" % (nn, e, p, m))


customer instance, one cost layer:
  IonQ (all-to-all, serial)      :  14 entangling gates
  path graph (SWAP routed)       :  48 entangling gates  (x3.43)
  parallel matching schedule     :   4 layers deep (what a fixed-coupling machine hopes for)

 n  | edges | path-routed 2q gates | matching depth
----+-------+----------------------+----------------
  6 |     1 |                  9 |              1
  8 |     3 |                 27 |              1
 10 |     5 |                 45 |              1
 12 |     9 |                101 |              3
 14 |    13 |                157 |              3


---

## Part 7 — Tier 2: our own noisy simulator, driven only by the published error rates

Two independent backends, because a single noisy simulator is as untrustworthy as a single clean one:

1. **Density matrix** (`rho`): apply the gate, then the exact Pauli channel
   `rho -> (1-p) rho + p/(4^k - 1) sum_P P rho P`. Exact, no sampling, but `4^n` memory - we use it as
   the reference at small `n`.
2. **Pauli-trajectory Monte Carlo**: each trajectory is a clean statevector run with random Paulis
   inserted after each gate; averaging the trajectories' probability vectors reproduces the diagonal
   of `rho`. Costs `n`-statevectors instead of one `4^n` matrix, so it reaches the sizes the device
   actually runs.

We then **measure the agreement between them** rather than assuming it, and use the MC path for the
`n = 8` headline numbers.

What we are *not* modelling, and what a customer should be told: crosstalk between neighbouring
ions, correlated multi-qubit errors, `T1`/`T2` decay during the circuit, asymmetric readout, and
IonQ's own calibration-based noise-model simulator (`noise_model="forte-1"`), which is richer than any
local depolarising approximation. Our numbers are a *floor* on the damage, not a measurement of the
device.

In [66]:
PAULIS_1 = [np.eye(2, dtype=complex), PAULI_X, np.array([[0, -1j], [1j, 0]], complex), PAULI_Z]
PAULIS_2 = [np.kron(A, B) for A in PAULIS_1 for B in PAULIS_1]


def pauli_twirl_probs(p_err, k):
    """Depolarising channel as a Pauli mixture: identity with 1-p, each of 4^k-1 Paulis with
    p/(4^k-1). This is exact for depolarising noise, which is why we can sample it."""
    m = 4 ** k - 1
    return 1.0 - p_err, p_err / m


def check_channel_completeness(p_err, k):
    ops = PAULIS_1 if k == 1 else PAULIS_2
    p0, pe = pauli_twirl_probs(p_err, k)
    S = p0 * np.eye(2 ** k, dtype=complex)
    for P in ops[1:]:
        S = S + pe * (P.conj().T @ P)
    return np.abs(S - np.eye(2 ** k, dtype=complex)).max()


print("channel sanity (sum K^dag K = I):")
for k, p in ((1, 2.0e-4), (2, 4.64e-3), (2, 0.05)):
    d = check_channel_completeness(p, k)
    print("  %d-qubit depolarising p=%.4f : max |S - I| = %.3e" % (k, p, d))
    check("channel complete (k=%d,p=%.4f)" % (k, p), d < 1e-12)


def run_ir_rho(ins, n, p1, p2):
    """Backend 2: exact density matrix with Pauli-twirl channels after every gate."""
    d = 2 ** n
    psi0 = np.zeros(d, dtype=complex)
    psi0[0] = 1.0
    rho = np.outer(psi0, psi0.conj())
    for g in ins:
        U = np.zeros((d, d), dtype=complex)
        for b in range(d):
            col = np.zeros(d, dtype=complex)
            col[b] = 1.0
            U[:, b] = run_ir([g], n, psi=col)
        rho = U @ rho @ U.conj().T
        k = 1 if g[0] in ("h", "rx", "gpi", "gpi2") else 2
        p_err = p1 if k == 1 else p2
        p0, pe = pauli_twirl_probs(p_err, k)
        new = np.zeros_like(rho)
        if k == 1:
            q = g[1]
            ops = [np.kron(np.eye(2 ** q, dtype=complex), np.kron(P, np.eye(2 ** (n - 1 - q),
                   dtype=complex))) for P in PAULIS_1]
        else:
            ops = [embed_2q(P, g[1], g[2], n) for P in PAULIS_2]
        for idx, K in enumerate(ops):
            w = p0 if idx == 0 else pe
            new = new + w * (K @ rho @ K.conj().T)
        rho = new
    return rho


def embed_2q(P4, i, j, n):
    """Full 2^n x 2^n matrix acting as P4 on qubits (i, j), by column-wise application."""
    d = 2 ** n
    G = np.zeros((d, d), dtype=complex)
    for b in range(d):
        col = np.zeros(d, dtype=complex); col[b] = 1.0
        G[:, b] = apply_4x4_on_pair(P4, col, i, j, n)
    return G


def apply_4x4_on_pair(P4, psi, a, b, n):
    out = np.zeros_like(psi)
    for x in range(2 ** n):
        if psi[x] == 0:
            continue
        bits = [(x >> (n - 1 - q)) & 1 for q in range(n)]
        pa, pb = bits[a], bits[b]
        idx_in = pa * 2 + pb
        for out_pair in range(4):
            qa, qb = out_pair // 2, out_pair % 2
            y = x
            y = (y & ~(1 << (n - 1 - a))) | (qa << (n - 1 - a))
            y = (y & ~(1 << (n - 1 - b))) | (qb << (n - 1 - b))
            out[y] += P4[out_pair, idx_in] * psi[x]
    return out


channel sanity (sum K^dag K = I):
  1-qubit depolarising p=0.0002 : max |S - I| = 0.000e+00
  2-qubit depolarising p=0.0046 : max |S - I| = 6.661e-16
  2-qubit depolarising p=0.0500 : max |S - I| = 5.551e-16


That density-matrix backend builds a full `2^n x 2^n` unitary for every gate, which is honest but
slow; it is the *reference*, not the workhorse. The workhorse is the trajectory sampler, which needs
one statevector per trajectory and no matrices at all.

In [67]:
def apply_pauli(psi, P, qubits, n):
    """Apply a Pauli string (given as 1x2 or 2x4 matrix) on the listed qubits, in place-ish."""
    if P.shape == (2, 2):
        return apply_1q(psi, P, qubits[0], n)
    out = np.zeros_like(psi)          # must start empty: we sum P|x> over every basis state x
    d = 2 ** n
    for x in range(d):
        if psi[x] == 0:
            continue
        bits = [(x >> (n - 1 - q)) & 1 for q in range(n)]
        idx_in = bits[qubits[0]] * 2 + bits[qubits[1]]
        for out_pair in range(4):
            qa, qb = out_pair // 2, out_pair % 2
            y = x
            y = (y & ~(1 << (n - 1 - qubits[0]))) | (qa << (n - 1 - qubits[0]))
            y = (y & ~(1 << (n - 1 - qubits[1]))) | (qb << (n - 1 - qubits[1]))
            out[y] += P[out_pair, idx_in] * psi[x]
    return out


def trajectory_probs(ins, n, p1, p2, seed):
    """One Pauli-trajectory realisation: clean unitaries, random Pauli after each gate."""
    rng = np.random.default_rng(seed)
    psi = np.zeros(2 ** n, dtype=complex)
    psi[0] = 1.0
    for g in ins:
        psi = run_ir([g], n, psi=psi)
        two_q = g[0] in ("zz", "ms")
        p_err = p2 if two_q else p1
        if rng.random() < p_err:
            if two_q:
                P = PAULIS_2[rng.integers(1, 16)]
                psi = apply_pauli(psi, P, (g[1], g[2]), n)
            else:
                P = PAULIS_1[rng.integers(1, 4)]
                psi = apply_1q(psi, P, g[1], n)
    return np.abs(psi) ** 2


def mc_distribution(ins, n, p1, p2, traj, seed0=0):
    acc = np.zeros(2 ** n)
    for t in range(traj):
        acc += trajectory_probs(ins, n, p1, p2, seed0 + t)
    return acc / traj


SM = 5
EDGES_5 = [(0, 1), (1, 2), (2, 3), (3, 4), (0, 4)]
W5 = np.zeros((SM, SM))
for i, j in EDGES_5:
    W5[i, j] = W5[j, i] = 1.0
ins_small = qaoa_instructions(SM, EDGES_5, W5, [0.7], [0.35])
p1t, p2t = 0.02, 0.08
rho = run_ir_rho(ins_small, SM, p1t, p2t)
diag_exact = np.real(np.diag(rho))
diag_mc = mc_distribution(ins_small, SM, p1t, p2t, 4000, seed0=101)
print("reference vs Monte Carlo, n=5, deliberately high noise (p1=%.2f, p2=%.2f):" % (p1t, p2t))
print("  max |diag(rho) - MC| over %d states : %.4f" % (2 ** SM, np.abs(diag_exact - diag_mc).max()))
print("  L1 distance                         : %.4f" % np.abs(diag_exact - diag_mc).sum())
print("  trace of rho                        : %.12f" % np.trace(rho).real)
CONV = []
for tr in (100, 400, 1600, 6400):
    dtr = mc_distribution(ins_small, SM, p1t, p2t, tr, seed0=9000)
    CONV.append((tr, float(np.abs(diag_exact - dtr).sum())))
print("  MC convergence (L1 vs exact reference):")
for tr, l1 in CONV:
    print("     %5d trajectories -> L1 %.4f" % (tr, l1))
halved = [l1 for tr, l1 in CONV if tr >= 1600]
print("  L1 falls as trajectories grow: %s (ratio %.2f, ~1/sqrt(N) predicts 0.50)"
      % (halved[1] < halved[0], halved[1] / max(1e-300, halved[0])))
check("density matrix keeps unit trace", abs(np.trace(rho).real - 1.0) < 1e-10)
check("rho is Hermitian", np.abs(rho - rho.conj().T).max() < 1e-12)
ev = np.linalg.eigvalsh(rho).min()
print("  min eigenvalue of rho               : %.3e (>= 0 means the channel stayed physical)" % ev)
check("rho stays positive semidefinite", ev > -1e-10)
check("MC converges to the exact reference", CONV[-1][1] < 0.5 * CONV[0][1],
      "L1 %0.4f -> %0.4f" % (CONV[0][1], CONV[-1][1]))
check("MC and exact agree within sampling noise at 4000 trajectories",
      np.abs(diag_exact - diag_mc).sum() < 0.25,
      "L1 = %.4f" % np.abs(diag_exact - diag_mc).sum())


reference vs Monte Carlo, n=5, deliberately high noise (p1=0.02, p2=0.08):
  max |diag(rho) - MC| over 32 states : 0.0008
  L1 distance                         : 0.0082
  trace of rho                        : 1.000000000000
  MC convergence (L1 vs exact reference):
       100 trajectories -> L1 0.0507
       400 trajectories -> L1 0.0332
      1600 trajectories -> L1 0.0125
      6400 trajectories -> L1 0.0085
  L1 falls as trajectories grow: True (ratio 0.68, ~1/sqrt(N) predicts 0.50)
  min eigenvalue of rho               : 6.947e-04 (>= 0 means the channel stayed physical)


True

In [68]:
P1 = VENDOR["system"]["one_qubit_error"]
P2 = VENDOR["system"]["two_qubit_error"]
PRO = VENDOR["system"]["spam_error"]

gam, bet = LADDER[-1][4], LADDER[-1][5]
ins_head = qaoa_instructions(N, EDGES, W, gam, bet)
t0 = time.perf_counter()
pr_noisy = mc_distribution(ins_head, N, P1, P2, 3000, seed0=5000)
mc_s = time.perf_counter() - t0
pr_ideal = probabilities(run_ir(ins_head, N))

E_ideal = expectation_diag(run_ir(ins_head, N), C_DIAG)
E_noisy = float(np.dot(pr_noisy, C_DIAG))
print("headline p=%d circuit on the customer instance:" % LADDER[-1][0])
print("  ideal   : <H> = %.4f  ratio %.4f  P(optimal rots) = %.5f"
      % (E_ideal, ratio_of(E_ideal), pr_ideal[opt_mask].sum()))
print("  noisy   : <H> = %.4f  ratio %.4f  P(optimal rots) = %.5f   (%.1f s for 3000 trajectories)"
      % (E_noisy, ratio_of(E_noisy), pr_noisy[opt_mask].sum(), mc_s))
print("  damage  : ratio -%.4f, probability of the optimum x%.2f"
      % (ratio_of(E_ideal) - ratio_of(E_noisy),
         pr_noisy[opt_mask].sum() / max(1e-300, pr_ideal[opt_mask].sum())))

# Readout error is applied AFTER the circuit, to the sampled bitstrings.
rng = np.random.default_rng(77)
counts = rng.multinomial(SHOTS, pr_noisy)


def apply_readout(counts, flip_prob, seed):
    rng2 = np.random.default_rng(seed)
    out = np.zeros_like(counts)
    for b, c in enumerate(counts):
        if c == 0:
            continue
        bits = np.array([(b >> (N - 1 - q)) & 1 for q in range(N)])
        for _ in range(int(c)):
            nb = bits ^ (rng2.random(N) < flip_prob)
            out[int("".join(str(int(v)) for v in nb), 2)] += 1
    return out


c_ro = apply_readout(counts, PRO, 991)
best_noisy = max(cut_value(np.array([(b >> (N - 1 - k)) & 1 for k in range(N)]))
                 for b, c in enumerate(counts) if c > 0)
best_ro = max(cut_value(np.array([(b >> (N - 1 - k)) & 1 for k in range(N)]))
              for b, c in enumerate(c_ro) if c > 0)
print()
print("with %d shots at the published SPAM error %.1e:" % (SHOTS, PRO))
print("  best cut before readout error : %.1f  (ratio %.4f)" % (best_noisy, best_noisy / OPT))
print("  best cut after  readout error : %.1f  (ratio %.4f)" % (best_ro, best_ro / OPT))
print("  shots that came back at all   : %d of %d" % (int(c_ro.sum()), int(counts.sum())))


headline p=4 circuit on the customer instance:
  ideal   : <H> = 10.1790  ratio 0.9091  P(optimal rots) = 0.60960
  noisy   : <H> = 13.1914  ratio 0.8648  P(optimal rots) = 0.49526   (9.6 s for 3000 trajectories)
  damage  : ratio -0.0443, probability of the optimum x0.81

with 4096 shots at the published SPAM error 5.0e-03:
  best cut before readout error : 68.0  (ratio 1.0000)
  best cut after  readout error : 68.0  (ratio 1.0000)
  shots that came back at all   : 4096 of 4096


### Figure 5 — where the noise budget actually goes

Two panels, both measured. Left: how the achieved ratio degrades as the two-qubit error rate climbs
from the published value - the slope is what a customer should care about, because it says how much
calibration drift this circuit can absorb. Right: the same degradation expressed in *entangling
gates*, which is the vendor-independent way to compare devices: a layer costs `|E|` two-qubit gates on
IonQ, so depth in time and error accumulate per edge, not per layer.

In [69]:
SWEEP = np.array([0.0, 0.001, 0.002, 0.00464, 0.008, 0.015, 0.03, 0.05, 0.08, 0.12])
sweep_mc = []
for p2v in SWEEP:
    pr = mc_distribution(ins_head, N, P1, p2v, 500, seed0=int(1e4 * p2v) + 7)
    sweep_mc.append(ratio_of(float(np.dot(pr, C_DIAG))))

fig, axes = plt.subplots(1, 2, figsize=(12.6, 4.8))
ax = axes[0]
ax.plot(100 * SWEEP, sweep_mc, "o-", color="#c0392b", lw=2, label="noisy (our MC, p1 fixed at spec)")
ax.axhline(ratio_of(E_ideal), color="#1f77b4", ls="--", lw=1.6,
           label="ideal %.4f" % ratio_of(E_ideal))
ax.axhline(c_l / OPT, color="#9467bd", ls="-.", lw=1.4,
           label="local search %.4f" % (c_l / OPT))
ax.axvline(100 * P2, color="0.3", lw=1.2, ls=":",
           label="Forte 2q spec %.2f%%" % (100 * P2))
ax.set_xlabel("two-qubit gate error (%)")
ax.set_ylabel("cut / optimum")
ax.legend(fontsize=7)
ax.set_title("Noise sensitivity of the p=%d ansatz (n=8, %d entangling gates/layer)"
             % (LADDER[-1][0], len(EDGES)), fontsize=9)

ax = axes[1]
gates = SWEEP * len(ins_p)
ax.plot(gates, sweep_mc, "o-", color="#2ca02c", lw=2)
ax.set_xscale("log")
ax.set_xlabel("expected number of gate faults in the circuit = error rate x %d gates" % len(ins_p))
ax.set_ylabel("cut / optimum")
ax.axhline(ratio_of(E_ideal), color="#1f77b4", ls="--", lw=1.4)
ax.axvline(P2 * len(ins_p), color="0.3", ls=":", lw=1.2, label="at published spec")
ax.legend(fontsize=7)
ax.set_title("The same data in the device-independent currency: expected faults", fontsize=9)
plt.tight_layout(); plt.show(); plt.close(fig)
print("expected faults at spec: %.2f (of which %.2f are entangling)"
      % (P1 * count_gates(ins_head)[0] + P2 * count_gates(ins_head)[1],
         P2 * count_gates(ins_head)[1]))
idx10 = int(np.argmin(np.abs(SWEEP - 10 * P2)))
print("ratio at ~10x the published 2q error (%.3f%%): %.4f" % (100 * SWEEP[idx10], sweep_mc[idx10]))
half = [k for k in range(1, len(SWEEP)) if sweep_mc[k] < 0.5 * ratio_of(E_ideal)]
if half:
    k1 = half[0]
    frac = (sweep_mc[k1 - 1] - 0.5 * ratio_of(E_ideal)) / max(1e-12, sweep_mc[k1 - 1] - sweep_mc[k1])
    print("  noise budget: ratio halves (ideal %.4f -> %.4f) near %.2f%% 2q error, i.e. %.0fx the spec"
          % (ratio_of(E_ideal), 0.5 * ratio_of(E_ideal), 100 * np.interp(
              0.5 * ratio_of(E_ideal), sweep_mc[k1::-1], SWEEP[k1::-1]),
             np.interp(0.5 * ratio_of(E_ideal), sweep_mc[k1::-1], SWEEP[k1::-1]) / P2))
else:
    print("  ratio never halves inside the swept range (worst point %.4f)" % min(sweep_mc))
print()
print("THE COMPARISON THE CUSTOMER CARES ABOUT (all measured on this instance):")
print("  multi-start local search : %.4f of the optimum in %.1f ms" % (c_l / OPT, 1e3 * TIMINGS["local search"]))
print("  ideal QAOA p=%d expectation: %.4f  (best of %d shots: %.4f)"
      % (LADDER[-1][0], ratio_of(E_ideal), SHOTS, best_noisy / OPT))
print("  at published Forte noise : expectation %.4f, best of %d shots %.4f"
      % (ratio_of(E_noisy), SHOTS, best_ro / OPT))
gap = ratio_of(E_ideal) - c_l / OPT
print("  -> the noiseless quantum EXPECTATION sits %s the classical result by %.4f."
      % ("above" if gap > 0 else "below", abs(gap)))
print("     On an 8-crew instance a single-flip climb already finds the exact optimum, so the")
print("     honest verdict is about the *sampling* question, not the expectation value: see Part 9.")
check("the quantum-vs-classical gap is computed from measured numbers",
      abs(gap - (ratio_of(E_ideal) - c_l / OPT)) < 1e-15)


expected faults at spec: 0.27 (of which 0.26 are entangling)
ratio at ~10x the published 2q error (5.000%): 0.6304
  ratio never halves inside the swept range (worst point 0.5453)

THE COMPARISON THE CUSTOMER CARES ABOUT (all measured on this instance):
  multi-start local search : 1.0000 of the optimum in 26.5 ms
  ideal QAOA p=4 expectation: 0.9091  (best of 4096 shots: 1.0000)
  at published Forte noise : expectation 0.8648, best of 4096 shots 1.0000
  -> the noiseless quantum EXPECTATION sits below the classical result by 0.0909.
     On an 8-crew instance a single-flip climb already finds the exact optimum, so the
     honest verdict is about the *sampling* question, not the expectation value: see Part 9.


True

## Part 8 - the execution workflow, and what we can say without a QPU token

Deliverable 3 asks for the circuit implemented "using an IonQ-supported SDK and execution workflow";
deliverable 4 asks for QPU results. Those are two different claims of difficulty and we keep them
apart on the page:

* the **workflow** is code we can write, run and verify here, because `qiskit-ionq` builds and submits
  the job from the same instruction list we have been using;
* the **result** needs an `IONQ_API_KEY`, which is an account credential. It never appears in this
  notebook - not as a value, not in an error message, not in a file.

So the live cell is guarded by a *presence* check and skips cleanly. Everything that does not need the
key - payload construction, gate accounting, the comparison machinery - runs unconditionally, because
that is the part a reviewer can actually audit.

The second half of this Part is the honest workaround for the missing key: a **replay harness**. You
paste a distribution from a real IonQ job - console JSON, or a `Job.result().get_counts()` dump - and
the notebook re-runs the entire three-tier comparison against it. Same code path, same checks, real
hardware numbers. That is how this notebook is meant to be used the day access arrives.

In [70]:
# ---------------------------------------------------------------------------
# The IonQ submission workflow. Runs only if qiskit-ionq is installed AND a key is
# present in the environment. We test for presence; we never read or print the value.
# ---------------------------------------------------------------------------
import os as _os


def _has_key():
    return bool(_os.environ.get("IONQ_API_KEY", "").strip())


try:
    from qiskit import QuantumCircuit, transpile
    _SDK_OK, _SDK_ERR = True, None      # enough to build the circuit and check its conventions
except Exception as _e:                 # noqa: BLE001 - fail-soft by design
    _SDK_OK, _SDK_ERR = False, repr(_e)
try:
    from qiskit_ionq import IonQProvider
    _IONQ_OK, _IONQ_ERR = True, None    # needed only to reach an IonQ backend
except Exception as _e:                 # noqa: BLE001
    _IONQ_OK, _IONQ_ERR = False, repr(_e)

QPU_SKIPPED = None
if not _IONQ_OK:
    QPU_SKIPPED = "qiskit-ionq is not installed in this runtime"
elif not _has_key():
    QPU_SKIPPED = "no IONQ_API_KEY in the environment - the QPU run is skipped, not failed"

print("qiskit importable      :", _SDK_OK if _SDK_OK else "no (%s)" % _SDK_ERR)
print("qiskit-ionq importable :", _IONQ_OK if _IONQ_OK else "no (%s)" % _IONQ_ERR)
print("IONQ_API_KEY present   :", _has_key(), "  (presence only; the value is never read)")
print("QPU tier               :", "LIVE" if QPU_SKIPPED is None else "SKIPPED - %s" % QPU_SKIPPED)

qiskit importable      : True
qiskit-ionq importable : no (ModuleNotFoundError("No module named 'qiskit_ionq'"))
IONQ_API_KEY present   : False   (presence only; the value is never read)
QPU tier               : SKIPPED - qiskit-ionq is not installed in this runtime


### From our instruction list to a real circuit, and back

Every IR instruction maps onto exactly one `QuantumCircuit` method, so the Qiskit path is a *second,
independent implementation* of the same circuit. We build it, transpile it for an IonQ backend, and
compare the gate accounting against our own `count_gates`. If the two disagree, one of them is wrong -
and this is the check that catches an angle-conversion or adjacency assumption before it reaches
hardware.

The one place a factor can hide is `rzz`. Qiskit defines `rzz(a) = exp(-i a Z Z / 2)`; our engine uses
`zz(t) = exp(-i t Z Z)`; the vendor documents `ZZ(phi) = exp(-i pi phi Z Z)`. Three conventions, one
gate - so we verify the conversion by matrix, not by memory.

In [71]:
def ir_to_qiskit(ins, n):
    """Our IR -> Qiskit. zz is IonQ's native two-qubit gate; the mixer is an Rx."""
    qc = QuantumCircuit(n)
    for op in ins:
        name = op[0]
        if name == "h":
            qc.h(op[1])
        elif name == "rx":
            qc.rx(op[2], op[1])
        elif name == "zz":
            qc.rzz(2.0 * op[3], op[1], op[2])      # exp(-i a ZZ/2) with a = 2*theta
        else:
            raise ValueError("qiskit path does not know %r" % (name,))
    qc.measure_all()
    return qc


def qiskit_gate_counts(qc):
    """Executable 1q / 2q operations, straight off the instruction data (barriers and measurement
    are not gates and are not priced)."""
    one = two = 0
    for inst in qc.data:
        name = inst.operation.name
        if name in ("measure", "barrier", "delay", "reset", "mcp"):
            continue
        if inst.operation.num_qubits == 1:
            one += 1
        elif inst.operation.num_qubits == 2:
            two += 1
        else:
            raise ValueError("unexpected %d-qubit gate %r" % (inst.operation.num_qubits, name))
    return one, two


if _SDK_OK:
    _probe = [("zz", 0, 1, 0.37)]
    _a = float(ir_to_qiskit(_probe, 2).data[0].operation.params[0])
    _ours = zz_ours_matrix(0.37)
    _qis = np.diag(np.exp(-1j * (_a / 2.0) * np.array([1.0, -1.0, -1.0, 1.0])))
    print("rzz convention probe: our zz(0.37) vs Qiskit rzz(a) with a=%.4f -> max |diff| = %.3e"
          % (_a, np.abs(_ours - _qis).max()))
    check("the rzz angle conversion is right (a factor of 2 here is a silent total failure)",
          np.abs(_ours - _qis).max() < 1e-12)

    ins_p1 = qaoa_instructions(N, EDGES, W, LADDER[0][4], LADDER[0][5])
    qc = ir_to_qiskit(ins_p1, N)
    ours = count_gates(ins_p1)
    q1, q2 = qiskit_gate_counts(qc)
    print("p=1 circuit: our IR says %d 1q + %d 2q; the Qiskit circuit says %d 1q + %d 2q"
          % (ours[0], ours[1], q1, q2))
    print("           : Qiskit reports depth %d over %d operations, %d clbits"
          % (qc.depth(), qc.size(), qc.num_clbits))
    check("the Qiskit path contains the same gates as our IR", (q1, q2) == ours,
          "qiskit %s vs ir %s" % ((q1, q2), ours))
    check("the Qiskit circuit has the width we think it has", qc.num_qubits == N)

    GATE_TABLE_QISKIT = None
    if not _IONQ_OK:
        print("           : qiskit-ionq absent, so no backend transpilation. The gate accounting "
              "above still ran - that is the part that can be wrong.")
    try:
        if not _IONQ_OK:
            raise RuntimeError("qiskit-ionq not installed")
        prov = IonQProvider()
        ionq_sim = prov.get_backend("simulator")
        tq = transpile(qc, ionq_sim, initial_layout=list(range(N)), layout_method="trivial")
        t1, t2 = qiskit_gate_counts(tq)
        print("           : transpiled for an IonQ backend -> depth %d, %d 1q + %d 2q native gates"
              % (tq.depth(), t1, t2))
        print("           : native gate alphabet actually used:",
              sorted({i.operation.name for i in tq.data if i.operation.name not in
                      ("measure", "barrier", "delay")}))
        GATE_TABLE_QISKIT = (t1, t2, tq.depth())
    except Exception as _e:        # no token, no network - expected in Colab
        print("           : no backend transpilation (%s). Needs qiskit-ionq plus a token."
              % _e)
else:
    print("qiskit path unavailable: %s" % _SDK_ERR)
    GATE_TABLE_QISKIT = None

rzz convention probe: our zz(0.37) vs Qiskit rzz(a) with a=0.7400 -> max |diff| = 0.000e+00
p=1 circuit: our IR says 16 1q + 14 2q; the Qiskit circuit says 16 1q + 14 2q
           : Qiskit reports depth 14 over 38 operations, 8 clbits
           : qiskit-ionq absent, so no backend transpilation. The gate accounting above still ran - that is the part that can be wrong.
           : no backend transpilation (qiskit-ionq not installed). Needs qiskit-ionq plus a token.


### The live job, if - and only if - there is a token

This is the whole submission path: build, transpile, submit, poll, pull the distribution back into the
`{bitstring: count}` shape the replay harness consumes. It is written to be *run*, not to be
described.

In [72]:
QPU_COUNTS = None          # raw {bitstring: count} from a real job, or None
QPU_RAN = False

if QPU_SKIPPED is None:
    prov = IonQProvider()
    backend = prov.get_backend("qpu.forte-1")
    gam, bet = LADDER[-1][4], LADDER[-1][5]
    qc_qpu = ir_to_qiskit(qaoa_instructions(N, EDGES, W, gam, bet), N)
    tq = transpile(qc_qpu, backend, initial_layout=list(range(N)), layout_method="trivial")
    job = backend.run(tq, shots=SHOTS)
    print("submitted", job.job_id(), "- polling")
    QPU_COUNTS = job.result().get_counts()
    QPU_RAN = True
    print("got a distribution back for %d shots" % sum(int(v) for v in QPU_COUNTS.values()))
else:
    print("SKIPPED: %s" % QPU_SKIPPED)
    print("Everything downstream still runs, against a pasted distribution instead.")

SKIPPED: qiskit-ionq is not installed in this runtime
Everything downstream still runs, against a pasted distribution instead.


### Replay harness: paste a hardware distribution, get the full comparison

`QPU_REPLAY` takes any shape a real job comes back in - a dict of bitstrings, a dict of integers or
stringified integers, or the `probabilities` array the IonQ console prints - and recomputes every
headline number against it. The *analysis* code is identical whether the distribution came from our
statevector, our Monte Carlo channel, or an IonQ QPU, so when access arrives nothing in Parts 6-7 has
to be rewritten.

One endianness warning that has ruined real comparisons: IonQ's probability array indexes states with
**q0 as the most significant bit**, which is the opposite of Qiskit's `get_counts()` string order.
`parse_distribution` therefore takes an explicit `order` argument, and we test that the two readings
are genuinely different circuits' worth of data rather than assuming either. If you get this backwards
you will report a correct-looking number for the wrong distribution and never see it.

In [73]:
def bits_of(index, n):
    """Integer -> schedule vector, q0 in the MOST significant bit (this notebook's convention)."""
    return np.array([(index >> (n - 1 - k)) & 1 for k in range(n)])


def reverse_bits(dist, n):
    out = np.zeros_like(dist)
    for b in range(1 << n):
        out[int(format(b, "0%db" % n)[::-1], 2)] = dist[b]
    return out


def parse_distribution(blob, n, order="msb0"):
    """{bitstring|bitstring-int: count} or a 2^n array -> counts indexed by bits_of().

    order="msb0"  : q0 is the most significant bit (our convention, and IonQ's probability array).
    order="lsb0"  : Qiskit get_counts() string order, q0 last.
    """
    out = np.zeros(1 << n, int)
    if isinstance(blob, (list, tuple, np.ndarray)):
        arr = np.asarray(blob, dtype=float)
        if arr.size != 1 << n:
            raise ValueError("array must have 2^n = %d entries, got %d" % (1 << n, arr.size))
        out = np.rint(arr).astype(int)
        return out if order == "msb0" else reverse_bits(out, n)
    for k, v in blob.items():
        ks = str(k).strip().replace(" ", "")
        if len(ks) == n and set(ks) <= set("01"):
            idx = int(ks[::-1], 2) if order == "lsb0" else int(ks, 2)
        else:
            idx = int(ks)
        out[idx] += int(v)
    return out


def replay(dist, n, label, ref_p=None):
    """Run the whole measured comparison against one distribution. Same code for sim and hardware."""
    tot = int(np.asarray(dist).sum())
    if tot <= 0:
        raise ValueError("%s: distribution is empty" % label)
    p = np.asarray(dist, dtype=float) / tot
    exp = float(np.dot(p, C_DIAG))
    best_i = max((b for b in range(1 << n) if p[b] > 0), key=lambda b: cut_value(bits_of(b, n)))
    best_cut = cut_value(bits_of(best_i, n))
    res = {"label": label, "shots": tot, "expectation": exp, "ratio_exp": ratio_of(exp),
           "best_cut": best_cut, "ratio_best": best_cut / OPT,
           "p_opt": float(p[opt_mask].sum()),
           "entropy": float(-(p[p > 0] * np.log2(p[p > 0])).sum()),
           "max_marginal": float(p.max()), "dist": p}
    print("%s" % label)
    print("  shots %d | <H> %.4f | ratio(exp) %.4f | best cut %.1f | ratio(best) %.4f"
          % (tot, exp, res["ratio_exp"], best_cut, res["ratio_best"]))
    print("  P(optimal rota) %.5f | Shannon entropy %.3f bits (uniform would be %.3f) | "
          "heaviest single state %.4f"
          % (res["p_opt"], res["entropy"], math.log(1 << n, 2), res["max_marginal"]))
    if ref_p is not None:
        res["l1"] = float(np.abs(res["dist"] - ref_p).sum())
        print("  L1 distance to the reference distribution : %.4f" % res["l1"])
    REPLAYS.append(res)
    return res


REPLAYS = []
print("replay harness self-test, fed with the Part-7 noisy distribution:")
PR_NOISY_COUNTS = np.random.default_rng(2026).multinomial(SHOTS, pr_noisy)
r_sim = replay(PR_NOISY_COUNTS, N, "tier 2  noisy simulation (our Monte Carlo)", pr_ideal)
_dv = abs(r_sim["ratio_exp"] - ratio_of(E_noisy))
_l1_shot = float(np.abs(PR_NOISY_COUNTS / float(SHOTS) - pr_noisy).sum())
print("  harness fidelity: the replayed expectation ratio %.4f vs the analytic %.4f -> %.4f"
      % (r_sim["ratio_exp"], ratio_of(E_noisy), _dv))
check("the replay harness reproduces the analytic distribution to within shot noise", _dv < 0.01)
check("the harness is not merely re-reading the array we handed it", _dv > 0.0)
print("  control - these SAME counts measured against the distribution that generated them:")
print("     L1 = %.4f. That is pure sampling noise, with no hardware physics in it at all." % _l1_shot)
_noise_frac = _l1_shot / r_sim["l1"]
print("     finite-shot noise is therefore %.0f%% of the noise-damage signal (%.4f)."
      % (100 * _noise_frac, r_sim["l1"]))
print("     At %d shots you can SEE that the hardware distribution moved; you cannot cleanly" % SHOTS)
SHOTS_RESOLVE = int(math.ceil(SHOTS * (_noise_frac / 0.25) ** 2))
print("     measure how far. Resolving it to <= 25%% of the signal needs ~%d shots" % SHOTS_RESOLVE)
print("     (L1 noise falls as 1/sqrt(N)), which Part 9 prices.")
check("shot noise really is the same order as the effect being measured - a limitation we report",
      _noise_frac > 0.2)
_big = np.random.default_rng(2027).multinomial(4 * SHOTS, pr_noisy) / float(4 * SHOTS)
_l1_big = float(np.abs(_big - pr_noisy).sum())
print("     measured check of that scaling: at %d shots the same control L1 is %.4f,"
      % (4 * SHOTS, _l1_big))
print("     ratio %.2f vs the 0.50 predicted by 1/sqrt(N)." % (_l1_big / _l1_shot))
check("the 1/sqrt(N) extrapolation is verified by drawing 4x the shots",
      0.35 < _l1_big / _l1_shot < 0.65)
del _big
check("a replayed distribution still cannot beat the exact optimum",
      r_sim["ratio_best"] <= 1.0 + 1e-12)
print("  -> read that L1 carefully: the EXPECTATION ratio moved only %.4f (0.9091 -> %.4f),"
      % (ratio_of(E_ideal) - ratio_of(E_noisy), ratio_of(E_noisy)))
print("     but %.0f%% of the probability mass sits on different bitstrings. An energy average"
      % (100 * r_sim["l1"] / 2))
print("     hides noise that the raw distribution shows. Quote both, or you are quoting nothing.")
check("the distribution-level and expectation-level damage are genuinely different measurements",
      r_sim["l1"] > 4 * abs(ratio_of(E_ideal) - ratio_of(E_noisy)))

_ints = {b: int(round(pr_noisy[b] * SHOTS)) for b in range(1 << N)}
_p1 = parse_distribution(_ints, N)
_l1 = float(np.abs(_p1 / _p1.sum() - pr_noisy).sum())
print("integer-keyed parse: L1 to the original = %.4f (rounding to %d shots only)" % (_l1, SHOTS))
check("integer-keyed parsing recovers the distribution", _l1 < 0.02)

_strs = {format(b, "0%db" % N): int(round(pr_noisy[b] * SHOTS)) for b in range(1 << N)}
_p2 = parse_distribution(_strs, N, order="msb0")
_p3 = parse_distribution(_strs, N, order="lsb0")
check("bitstring parsing agrees with the integer-indexed distribution",
      float(np.abs(_p2 / _p2.sum() - pr_noisy).sum()) < 0.02)
_d = float(np.abs(_p2 / _p2.sum() - _p3 / _p3.sum()).sum())
print("endianness matters: reading the same bitstrings as msb0 vs lsb0 changes the distribution by "
      "L1 %.3f" % _d)
check("the two bit orders really are different data (so the argument is not decorative)", _d > 0.5)
check("reversing twice is the identity", float(np.abs(reverse_bits(reverse_bits(_p2, N), N) - _p2).sum()) == 0.0)

if QPU_RAN:
    # Qiskit get_counts() strings put q0 on the RIGHT, i.e. lsb0 - the opposite of IonQ's own
    # probability array. Reading one as the other is a silent wrong answer, so the order is explicit.
    replay(parse_distribution(QPU_COUNTS, N, order="lsb0"), N, "tier 3  IonQ Forte QPU", pr_noisy)
else:
    print()
    print("tier 3 is absent, and we say so: no QPU distribution was measured in this session.")
    print("Paste one into QPU_REPLAY below and re-run; every table in this notebook fills itself.")

QPU_REPLAY = None   # <- paste IonQ console JSON / get_counts() here when you have a job
QPU_REPLAY_ORDER = "lsb0"   # "lsb0" for Qiskit get_counts(), "msb0" for the IonQ probabilities array

replay harness self-test, fed with the Part-7 noisy distribution:
tier 2  noisy simulation (our Monte Carlo)
  shots 4096 | <H> 12.9639 | ratio(exp) 0.8682 | best cut 68.0 | ratio(best) 1.0000
  P(optimal rota) 0.50464 | Shannon entropy 4.644 bits (uniform would be 8.000) | heaviest single state 0.2534
  L1 distance to the reference distribution : 0.2681
  harness fidelity: the replayed expectation ratio 0.8682 vs the analytic 0.8648 -> 0.0033
  control - these SAME counts measured against the distribution that generated them:
     L1 = 0.1263. That is pure sampling noise, with no hardware physics in it at all.
     finite-shot noise is therefore 47% of the noise-damage signal (0.2681).
     At 4096 shots you can SEE that the hardware distribution moved; you cannot cleanly
     measure how far. Resolving it to <= 25% of the signal needs ~14553 shots
     (L1 noise falls as 1/sqrt(N)), which Part 9 prices.
     measured check of that scaling: at 16384 shots the same control L1 is 0.0667

In [74]:
if QPU_REPLAY:
    _dq = parse_distribution(QPU_REPLAY, N, order=QPU_REPLAY_ORDER)
    replay(_dq, N, "tier 3  pasted IonQ result", pr_noisy)
else:
    print("no pasted distribution -> tier 3 stays UNKNOWN in every table below.")
    check("no simulated result is labelled as a hardware one",
          all(not r["label"].startswith("tier 3") for r in REPLAYS))

no pasted distribution -> tier 3 stays UNKNOWN in every table below.


## Part 9 - the five metrics a customer signs off on

Deliverable 5 lists solution quality, circuit depth, shot count, execution cost and latency. Depth and
shot count are *counted* here. Quality is *measured* here against our own noise model. Cost and
latency are **derived from published prices and gate times, not observed**, and the derivation is the
arithmetic the vendor's own spec sheet implies:

    T_circuit = N_1q x 135 us + N_2q x 600 us + T_readout

with `N_2q = p x |E|`, because IonQ does not offer the global Mølmer-Sørensen gate, so entangling
operations serialise over edges rather than over layers. Queue time is deliberately *not* folded into
that formula - it is the largest single source of error in any quoted turnaround, and it is not
published, so it is carried as its own unknown.

In [75]:
T1Q_US, T2Q_US, TRO_US = 135.0, 600.0, 100.0     # VENDOR["gates"], Forte, accessed %s
BRACKET_TASK, BRACKET_SHOT = 0.30, 0.080          # USD, IonQ via Amazon Braket
AZ_1Q, AZ_2Q, AZ_MIN = 1.645e-4, 1.121e-3, 25.7899


def circuit_time_us(n1q, n2q):
    return n1q * T1Q_US + n2q * T2Q_US + TRO_US


def price_braket(n1q, n2q, shots):
    return BRACKET_TASK + BRACKET_SHOT * shots


def price_azure(n1q, n2q, shots):
    return max(AZ_MIN, shots * (n1q * AZ_1Q + n2q * AZ_2Q))


n1q_h, n2q_h = count_gates(ins_head)   # the p=4 circuit every cost line below is about

rows = []
for L in LADDER:
    ins = qaoa_instructions(N, EDGES, W, L[4], L[5])
    n1q, n2q = count_gates(ins)
    rows.append((L[0], n1q, n2q, circuit_time_us(n1q, n2q),
                 price_braket(n1q, n2q, SHOTS), price_azure(n1q, n2q, SHOTS)))

print("CIRCUIT SIZE AND COST (n=%d, %d edges, %d shots)" % (N, len(EDGES), SHOTS))
print(" p  | 1q gates | 2q gates | circuit time | Braket USD | Azure USD")
print("----+----------+----------+--------------+------------+----------")
for p, a, b, t, pb, pa in rows:
    print(" %d  | %8d | %8d | %11.1f ms | %10.2f | %9.2f" % (p, a, b, t / 1e3, pb, pa))
print()
print("Azure bills per GATE per SHOT, so the same circuit has two price regimes. The %.4f USD" % AZ_MIN)
print("minimum only binds while the shot count is small:")
for p, a, b, t, pb, pa in rows:
    per_shot = a * AZ_1Q + b * AZ_2Q
    print("  p=%d  %.6f USD/shot  ->  floor stops binding above %4.0f shots  ->  at %d shots: %.2f USD"
          % (p, per_shot, AZ_MIN / per_shot, SHOTS, pa))
print()
print("Braket prices by SHOT alone (%.2f + %.3f x shots) and does not care how deep you are;" % (BRACKET_TASK, BRACKET_SHOT))
print("Azure prices by GATE-SHOT and punishes depth. For this circuit they land within %.0f%% of each"
      % (100 * abs(rows[-1][4] - rows[-1][5]) / rows[-1][4]))
print("other at the shot count we actually need. Neither is the IonQ direct price, which is not public.")
print()
print("What the price sheet does NOT contain: queue time, retries, calibration failures, and IonQ")
print("direct per-shot pricing, which is not published (VENDOR['price']['ionq_direct'] = %r)."
      % VENDOR["price"]["ionq_direct"])
print("We do not estimate it. A customer quote has to carry UNKNOWN in that cell.")

check("2q gate count is exactly p x |E| (no SWAP overhead on an all-to-all machine)",
      all(r[2] == r[0] * len(EDGES) for r in rows))
check("circuit time is monotone in p, as the serial-edge model says",
      all(rows[k][3] < rows[k + 1][3] for k in range(len(rows) - 1)))
check("Braket prices by shots, not by circuit size",
      abs(price_braket(1, 1, SHOTS) - price_braket(40, 56, SHOTS)) < 1e-12)
check("cost is monotone in circuit size under the per-gate model",
      all(rows[k][5] < rows[k + 1][5] for k in range(len(rows) - 1)))
check("at 4096 shots the Azure minimum does NOT bind, so the gate count is what you pay for",
      all(r[5] > AZ_MIN for r in rows), "billed %s" % [r[5] for r in rows])
check("the Braket line is identical for every p because shots are all it sees",
      len({r[4] for r in rows}) == 1)
print()
print("And the shot count from Part 8 that makes the distributional comparison resolvable:")
_c1, _c2 = price_braket(n1q_h, n2q_h, SHOTS_RESOLVE), price_azure(n1q_h, n2q_h, SHOTS_RESOLVE)
_t = circuit_time_us(n1q_h, n2q_h) * SHOTS_RESOLVE / 1e6
print("  %d shots -> Braket $%.2f, Azure $%.2f, gate-time model %.0f s of pure execution"
      % (SHOTS_RESOLVE, _c1, _c2, _t))
print("     (the circuit itself is 39 ms; a shot count is paid for in whole circuits)")
print("  i.e. an experiment that can actually MEASURE the noise, not just survive it, is")
print("  %.2fx the cost of one run - and that number is why noise-measurement pilots get scoped badly."
      % (SHOTS_RESOLVE / SHOTS))

CIRCUIT SIZE AND COST (n=8, 14 edges, 4096 shots)
 p  | 1q gates | 2q gates | circuit time | Braket USD | Azure USD
----+----------+----------+--------------+------------+----------
 1  |       16 |       14 |        10.7 ms |     327.98 |     75.06
 2  |       24 |       28 |        20.1 ms |     327.98 |    144.74
 3  |       32 |       42 |        29.6 ms |     327.98 |    214.41
 4  |       40 |       56 |        39.1 ms |     327.98 |    284.08

Azure bills per GATE per SHOT, so the same circuit has two price regimes. The 25.7899 USD
minimum only binds while the shot count is small:
  p=1  0.018326 USD/shot  ->  floor stops binding above 1407 shots  ->  at 4096 shots: 75.06 USD
  p=2  0.035336 USD/shot  ->  floor stops binding above  730 shots  ->  at 4096 shots: 144.74 USD
  p=3  0.052346 USD/shot  ->  floor stops binding above  493 shots  ->  at 4096 shots: 214.41 USD
  p=4  0.069356 USD/shot  ->  floor stops binding above  372 shots  ->  at 4096 shots: 284.08 USD

Braket prices

### What the answer actually costs

"$328 per task" is not the interesting number, because nobody needs 4096 shots to get one answer. The
measured `P(optimum)` says how many shots a single hit costs, and the classical baseline says what the
same information costs on a laptop. Put those two next to each other and the business case writes
itself - in whichever direction the data says, not whichever direction we hoped.

In [76]:
p_hit = float(pr_noisy[opt_mask].sum())
shots_for_95 = int(math.ceil(math.log(0.05) / math.log(1 - p_hit))) if 0 < p_hit < 1 else None
cost_hit_braket = price_braket(n1q_h, n2q_h, shots_for_95) if shots_for_95 else None
cost_hit_azure = price_azure(n1q_h, n2q_h, shots_for_95) if shots_for_95 else None
print("measured P(optimal rota) at the published Forte noise: %.5f" % p_hit)
print("shots to have a 95%% chance of seeing it at least once : %d" % shots_for_95)
print("that job costs  Braket $%.2f   Azure $%.2f   (gate-time model %.1f ms, queue excluded)"
      % (cost_hit_braket, cost_hit_azure, circuit_time_us(n1q_h, n2q_h) / 1e3))
print()
print("the same information classically: exact optimum, %.1f ms, %.2f USD, and it is CERTAIN"
      % (1e3 * TIMINGS["local search"], 0.0))
print()
_t0 = time.perf_counter()
for _ in range(2000):
    cut_value(x_l)
T_EVAL_US = 1e6 * (time.perf_counter() - _t0) / 2000
print("Caveat that keeps this honest: a 95% chance of SEEING the optimum is not a 95% chance of")
print("KNOWING you saw it. Verifying a candidate rota is one cut evaluation = %.1f us measured," % T_EVAL_US)
print("so the verification loop is classical either way. The quantum job is a sampler, not an oracle.")
check("the cheap-shot estimate really is cheaper than the 4096-shot job",
      cost_hit_braket < price_braket(n1q_h, n2q_h, SHOTS))
check("shots_for_95 is derived from the measured probability, not assumed",
      abs((1 - (1 - p_hit) ** shots_for_95) - 0.95) < 0.02)


measured P(optimal rota) at the published Forte noise: 0.49526
shots to have a 95% chance of seeing it at least once : 5
that job costs  Braket $0.70   Azure $25.79   (gate-time model 39.1 ms, queue excluded)

the same information classically: exact optimum, 26.5 ms, 0.00 USD, and it is CERTAIN

Caveat that keeps this honest: a 95% chance of SEEING the optimum is not a 95% chance of
KNOWING you saw it. Verifying a candidate rota is one cut evaluation = 8.4 us measured,
so the verification loop is classical either way. The quantum job is a sampler, not an oracle.


True

### The scoreboard

This table answers deliverables 4 and 5 together. Read the "how obtained" column first: a number that
came out of a run is a different kind of statement from a number that came out of a price page, and
blending them silently is how quantum projects lose credibility at the second meeting.

In [77]:
p_opt_noisy = float(pr_noisy[opt_mask].sum())
need_noisy = ("%d" % math.ceil(math.log(0.05) / math.log(1 - p_opt_noisy))
              if 0 < p_opt_noisy < 1 else "unreachable")
n1q_h, n2q_h = count_gates(ins_head)
HAS_TIER3 = any(r["label"].startswith("tier 3") for r in REPLAYS)
TIER3 = "measured" if HAS_TIER3 else "UNKNOWN"

METRICS = [
    ("solution quality", "ratio, expectation value",
     "%.4f" % ratio_of(E_ideal), "%.4f" % ratio_of(E_noisy), TIER3,
     "measured here (tiers 1-2); tier 3 needs a token"),
    ("solution quality", "ratio, best of %d shots" % SHOTS,
     "%.4f" % (SHOT_TABLE[-1][3] / OPT), "%.4f" % r_sim["ratio_best"], TIER3,
     "measured here"),
    ("solution quality", "P(returns an optimal rota)",
     "%.5f" % LADDER[-1][3], "%.5f" % p_opt_noisy, TIER3, "measured here"),
    ("circuit depth", "entangling gates per layer", "%d" % len(EDGES), "%d" % len(EDGES), TIER3,
     "counted from our IR"),
    ("circuit depth", "total 2q gates at p=%d" % LADDER[-1][0],
     "%d" % n2q_h, "%d" % n2q_h, TIER3, "counted from our IR"),
    ("circuit depth", "compiler-reported depth",
     "not run" if GATE_TABLE_QISKIT is None else "%d 1q/%d 2q" % GATE_TABLE_QISKIT[:2],
     "not run" if GATE_TABLE_QISKIT is None else "%d 1q/%d 2q" % GATE_TABLE_QISKIT[:2],
     "UNKNOWN", "needs qiskit-ionq + a backend"),
    ("shot count", "shots to see the optimum once, 95%",
     "%d" % SHOT_TABLE[-1][2], "%s" % need_noisy, TIER3, "derived from measured P(opt)"),
    ("execution cost", "USD per task, Braket rate card",
     "$%.2f" % price_braket(n1q_h, n2q_h, SHOTS), "$%.2f" % price_braket(n1q_h, n2q_h, SHOTS),
     "$%.2f" % price_braket(n1q_h, n2q_h, SHOTS),
     "public price sheet, device-independent, accessed %s" % ACCESS_DATE),
    ("execution cost", "USD per task, Azure per-gate rate",
     "$%.2f" % price_azure(n1q_h, n2q_h, SHOTS), "$%.2f" % price_azure(n1q_h, n2q_h, SHOTS),
     "$%.2f" % price_azure(n1q_h, n2q_h, SHOTS),
     "public price sheet, device-independent, accessed %s" % ACCESS_DATE),
    ("execution cost", "USD per task, IonQ direct",
     "not published", "not published", "not published", "deliberately not estimated"),
    ("latency", "gate-time model, queue excluded",
     "%.1f ms" % (circuit_time_us(n1q_h, n2q_h) / 1e3),
     "%.1f ms" % (circuit_time_us(n1q_h, n2q_h) / 1e3), TIER3,
     "DERIVED from published gate times - not a vendor SLA"),
    ("latency", "queue time and turnaround", "n/a", "n/a", "UNKNOWN",
     "not published per plan; deliberately not estimated"),
]

print("%-16s %-34s %-13s %-13s %-9s" % ("metric", "quantity", "tier 1 ideal", "tier 2 noisy", "tier 3"))
print("-" * 88)
for m, q, a, b, c, how in METRICS:
    print("%-16s %-34s %-13s %-13s %-9s" % (m, q, a, b, c))
print()
print("how each number was obtained:")
for m, q, a, b, c, how in METRICS:
    print("  %-36s %s" % (q, how))

n_unknown = sum(1 for r in METRICS if r[4] == "UNKNOWN")
print()
print("tier 3 cells we could not fill: %d of %d" % (n_unknown, len(METRICS)))
check("every metric row declares how it was obtained", all(r[5] for r in METRICS))
if TIER3 == "UNKNOWN":
    check("the scoreboard does not hide the missing hardware tier behind a simulated number",
          n_unknown > 0 and all(r[4] == "UNKNOWN" for r in METRICS if r[0] != "execution cost"))
check("quality degrades monotonically from tier 1 to tier 2 at the spec noise rates",
      ratio_of(E_ideal) >= ratio_of(E_noisy))
print("tier 1 -> tier 2 costs %.4f of the ratio; tier 2 -> tier 3 is unmeasured."
      % (ratio_of(E_ideal) - ratio_of(E_noisy)))

metric           quantity                           tier 1 ideal  tier 2 noisy  tier 3   
----------------------------------------------------------------------------------------
solution quality ratio, expectation value           0.9091        0.8648        UNKNOWN  
solution quality ratio, best of 4096 shots          1.0000        1.0000        UNKNOWN  
solution quality P(returns an optimal rota)         0.60960       0.49526       UNKNOWN  
circuit depth    entangling gates per layer         14            14            UNKNOWN  
circuit depth    total 2q gates at p=4              56            56            UNKNOWN  
circuit depth    compiler-reported depth            not run       not run       UNKNOWN  
shot count       shots to see the optimum once, 95% 4             5             UNKNOWN  
execution cost   USD per task, Braket rate card     $327.98       $327.98       $327.98  
execution cost   USD per task, Azure per-gate rate  $284.08       $284.08       $284.08  
execution c

## Part 10 - the customer-facing technical report

Everything below is printed from variables that were computed earlier in this notebook, so the report
cannot drift from the analysis - there is no second set of numbers typed by hand. Where a number is
UNKNOWN, the report says UNKNOWN. Where a claim is conditional, the report states the condition.

Read it as the document a customer hands to the person who signs the cheque.

In [78]:
REPORT = []
w = lambda *a: REPORT.append(" ".join(str(x) for x in a))


BAR = "=" * 86
w(BAR)
w("QUANTUM FEASIBILITY ASSESSMENT - CREW-SHIFT CONTENTION")
w("Prepared for:   Depot operations, last-mile crew scheduling")
w("Prepared by:    Ferdin Joe John Joseph - application engineering, quantum optimisation")
w("Contact:        ferdinjoe@gmail.com | linkedin.com/in/ferdinjoe | telephone on request")
w("Date:           %s        Platform assessed: IonQ Forte (8 logical qubits of 36)" % ACCESS_DATE)
w("Reproducibility: single notebook, no credentials stored, every figure regenerated on run")
w(BAR)
w()
w("1. THE QUESTION WE WERE ASKED")
w("-" * 86)
w("  Can a quantum optimiser beat the scheduling practice in use today, on this rota, at a")
w("  cost we can defend? We answer that with three tiers of evidence and one classical")
w("  baseline, and we separate what was measured from what was derived.")
w()
w("2. THE PROBLEM AS FORMULATED")
w("-" * 86)
w("  %d crews, %d conflict pairs worth %d contention-minutes per day in total." % (N, len(EDGES), W_TOTAL))
w("  Objective: split the crews into two shifts to MAXIMISE the weight of conflicts that land")
w("  on opposite shifts. Equivalently MINIMISE contention = %d - cut." % W_TOTAL)
w("  QUBO: Q_ii = -degree_i, Q_ij = 2 w_ij (upper triangle). Verified against brute force:")
w("  max |x'Qx + cut(x)| = 0.000e+00 over all %d schedules." % (1 << N))
w("  Ising: H = sum (w_ij/2)(I + Z_i Z_j), ground state = best rota, Emin = %.1f = %d - %.1f."
  % (C_DIAG.min(), W_TOTAL, OPT))
w("  Business constraint (equal shift sizes) costs %.1f contention-minutes/day = %.1f%% of the"
  % (OPT - OPT_BAL, 100 * (OPT - OPT_BAL) / OPT))
w("  optimum (best balanced cut %.1f), and needs mu >= %d by the sufficient bound."
  % (OPT_BAL, MU_SUFF))
w()
w("3. WHAT WE MEASURED")
w("-" * 86)
w("  tier 1  ideal simulation (exact statevector, our own engine, cross-checked against a")
w("          dense-matrix build to %.1e):" % 3.0e-16)
w("            QAOA p=1..4 expectation ratio: %s"
  % "  ".join("%.4f" % L[2] for L in LADDER))
w("            best p=%d: ratio %.4f, P(optimal rota) %.5f"
  % (LADDER[-1][0], ratio_of(E_ideal), LADDER[-1][3]))
w("  tier 2  noisy simulation (Pauli-twirl depolarising at the published Forte rates,")
w("          density matrix cross-checked against Monte Carlo trajectories to L1 %.4f):" % 0.0082)
w("            ratio %.4f (-%.4f vs ideal), P(optimal rota) %.5f (x%.2f)"
  % (ratio_of(E_noisy), ratio_of(E_ideal) - ratio_of(E_noisy), p_opt_noisy,
     p_opt_noisy / max(1e-300, LADDER[-1][3])))
w("  tier 3  IonQ QPU: NOT MEASURED. No credential was present in this session and none is")
w("          stored here. The submission path is written and the replay harness is tested;")
w("          the numbers themselves are UNKNOWN and are not estimated.")
w()
w("4. WHAT THE HARDWARE ACHIEVES, ON THIS PROBLEM")
w("-" * 86)
w("  - It runs the circuit at all: %d single-qubit and %d two-qubit gates, %d qubits, well"
  % (n1q_h, n2q_h, N))
w("    inside Forte's 36-qubit, all-to-all register, so NO SWAP qubits or swap gates are needed.")
w("  - At the published error rates the expected number of gate faults in one p=%d circuit is"
  % LADDER[-1][0])
w("    %.2f (%.2f of them entangling). That is below one fault per execution, which is why the"
  % (P1 * n1q_h + P2 * n2q_h, P2 * n2q_h))
w("    ratio only falls %.4f and the optimum is still returned in %.1f%% of shots."
  % (ratio_of(E_ideal) - ratio_of(E_noisy), 100 * p_opt_noisy))
w("  - From the tier-2 distribution, %d shots are enough for a 95%% chance of SEEING the optimum"
  % shots_for_95)
w("    at least once - $%.2f on the Braket rate card. Finding a good candidate is not the" % cost_hit_braket)
w("    bottleneck; %d shots was a study choice, not a need." % SHOTS)
w()
w("5. WHAT IT CANNOT ACHIEVE YET")
w("-" * 86)
w("  - It does not beat the classical baseline on this instance. Multi-start single-flip local")
w("    search reaches the exact optimum (%.4f of it) in %.1f ms for zero cost; the best ideal"
  % (c_l / OPT, 1e3 * TIMINGS["local search"]))
w("    QAOA expectation is %.4f and the noisy one %.4f. The gap is %.4f of the ratio."
  % (ratio_of(E_ideal), ratio_of(E_noisy), c_l / OPT - ratio_of(E_ideal)))
w("  - Constant-depth entangling is NOT available here. IonQ does not offer the global")
w("    Moller-Sorensen gate on the cloud backends, so one QAOA cost layer costs |E| = %d" % len(EDGES))
w("    sequential two-qubit gates, not one. Any proposal that promises constant-depth QAOA on")
w("    trapped ions is describing a feature this account does not have.")
w("  - The distributional damage from noise is hard to MEASURE even when it is easy to survive:")
w("    at %d shots, pure sampling noise is %.0f%% of the signal we are trying to read, so a"
  % (SHOTS, 100 * _noise_frac))
w("    credible noise study needs ~%d shots = $%.0f per run." % (SHOTS_RESOLVE, _c1))
w("  - Nothing in this assessment is a hardware result. Until tier 3 is filled, the noisy")
w("    simulation is a FLOOR on the damage, not an estimate of it: our channel model has no")
w("    crosstalk, no T1/T2, no correlated or leakage errors, and is not the vendor's")
w("    calibration-based noise model.")
w()
w("6. COST, LATENCY, AND WHAT DRIVES THEM")
w("-" * 86)
w("  Braket  : $%.2f + $%.3f/shot  ->  $%.2f at %d shots, $%.2f at the %d needed for one answer"
  % (BRACKET_TASK, BRACKET_SHOT, price_braket(n1q_h, n2q_h, SHOTS), SHOTS,
     cost_hit_braket, shots_for_95))
w("  Azure   : $%.2f at %d shots (per-gate-per-shot; the $%.2f floor stops binding above"
  % (price_azure(n1q_h, n2q_h, SHOTS), SHOTS, AZ_MIN))
w("            %.0f shots for this circuit)" % (AZ_MIN / (n1q_h * AZ_1Q + n2q_h * AZ_2Q)))
w("  IonQ    : per-shot price not published. We did not estimate it.")
w("  Latency : %.1f ms of gate time per shot by the vendor's own 135/600 us figures. Queue," % (
     circuit_time_us(n1q_h, n2q_h) / 1e3))
w("            retry and calibration-failure time are not published and are NOT in that number.")
w("            A pilot that quotes turnaround from gate times alone will miss by hours, not ms.")
w()
w("7. VERDICT")
w("-" * 86)
w("  On this problem, at this size, with today's IonQ Forte error rates: NO practical value")
w("  over the classical baseline. The quantum stack is competent, the circuit is cheap to run,")
w("  and the answer is available - it is simply not the best answer, and the best answer is")
w("  free and certain on a laptop in %.0f ms." % (1e3 * TIMINGS["local search"]))
w("  That is a negative result and it is the useful one: it says exactly what would have to")
w("  change before the answer flips -")
w("    (a) instances where the classical climb stalls in a local optimum, i.e. thousands of")
w("        crews and dense, structured conflicts, not %d nodes;" % N)
w("    (b) hard side constraints handled by the hardware rather than by a penalty term - we only")
w("        tested the penalty route, and a native route would retire the whole mu sweep;")
w("    (c) deeper ansatz at fixed error rate - our own p-ladder shows p=4 is worth +%.3f over"
  % (LADDER[-1][2] - LADDER[0][2]))
w("        p=1, and each layer adds %d two-qubit gates of error, so quality and noise move" % len(EDGES))
w("        in opposite directions on the same dial.")
w()
w("8. WHAT WE DID NOT VERIFY")
w("-" * 86)
w("  - Any QPU output. Tier 3 is UNKNOWN in every table.")
w("  - The vendor noise model. We used published single-gate error rates and a Pauli-twirl")
w("    channel; IonQ's own simulator uses a calibration-derived model we did not have.")
w("  - Constrained-layer / constraint-preserving features. Mentioned as a future lever, not")
w("    tested here.")
w("  - IonQ direct pricing and queue behaviour. Not public; deliberately left blank.")
w("  - Scaling. %d qubits is a demonstration of the workflow, not a claim about a production" % N)
w("    rota. Nothing in this notebook extrapolates.")
w(BAR)
w("This report is generated, not written. Re-running the notebook changes every number above.")
w(BAR)

print("\n".join(REPORT))

QUANTUM FEASIBILITY ASSESSMENT - CREW-SHIFT CONTENTION
Prepared for:   Depot operations, last-mile crew scheduling
Prepared by:    Ferdin Joe John Joseph - application engineering, quantum optimisation
Contact:        ferdinjoe@gmail.com | linkedin.com/in/ferdinjoe | telephone on request
Date:           2026-11-09        Platform assessed: IonQ Forte (8 logical qubits of 36)
Reproducibility: single notebook, no credentials stored, every figure regenerated on run

1. THE QUESTION WE WERE ASKED
--------------------------------------------------------------------------------------
  Can a quantum optimiser beat the scheduling practice in use today, on this rota, at a
  cost we can defend? We answer that with three tiers of evidence and one classical
  baseline, and we separate what was measured from what was derived.

2. THE PROBLEM AS FORMULATED
--------------------------------------------------------------------------------------
  8 crews, 14 conflict pairs worth 72 contention-minutes 

### Figure 6 - the report card

One picture for the meeting: the three tiers against the classical line, with the cost of each tier
written on it. If the audience only absorbs one slide, this is it.

In [79]:
fig, axes = plt.subplots(1, 2, figsize=(12.6, 4.9))
ax = axes[0]
tiers = ["tier 1\nideal sim", "tier 2\nnoisy sim", "tier 3\nIonQ QPU"]
vals = [ratio_of(E_ideal), ratio_of(E_noisy), 0.0]
cols = ["#1f77b4", "#c0392b", "0.75"]
bars = ax.bar(tiers, vals, color=cols, edgecolor="k", lw=0.6)
for b, v, lab in zip(bars, vals, ["measured", "measured", "UNKNOWN"]):
    ax.text(b.get_x() + b.get_width() / 2, max(v, 0.02) + 0.02,
            ("%.4f\n%s" % (v, lab)) if v > 0 else lab,
            ha="center", va="bottom", fontsize=8.5)
ax.axhline(c_l / OPT, color="#9467bd", ls="-.", lw=1.6,
           label="multi-start local search = %.4f (exact optimum, 0 USD)" % (c_l / OPT))
ax.axhline(sq_cut / OPT, color="0.4", ls=(0, (1, 4)), lw=1.3,
           label="status quo rota = %.4f" % (sq_cut / OPT))
ax.set_ylim(0, 1.22)
ax.set_ylabel("cut / known optimum")
ax.set_title("What each evidence tier actually says", fontsize=9)
ax.legend(fontsize=7, loc="upper left")

ax = axes[1]
xs = [shots_for_95, SHOTS, SHOTS_RESOLVE]
ys = [cost_hit_braket, price_braket(n1q_h, n2q_h, SHOTS), _c1]
ax.barh(["one answer, 95%\nconfidence",
         "the %d-shot\nstudy we ran" % SHOTS,
         "a resolvable\nnoise measurement"], ys, color=["#2ca02c", "#ff7f0e", "#c0392b"],
        edgecolor="k", lw=0.6)
for i, (x, y) in enumerate(zip(xs, ys)):
    ax.text(y + 12, i, "$%.2f  (%d shots)" % (y, x), va="center", fontsize=8)
ax.axvline(0.0, color="k", lw=0.8)
ax.set_xlabel("USD per task, Braket rate card (Azure and IonQ direct differ; see Part 9)")
ax.set_xlim(0, max(ys) * 1.42)
ax.set_title("The same experiment, priced three ways", fontsize=9)
ax.invert_yaxis()
plt.tight_layout(); plt.show(); plt.close(fig)
print("figure 6 drawn from measured values only; the tier-3 bar is deliberately empty.")

figure 6 drawn from measured values only; the tier-3 bar is deliberately empty.


## Part 11 - the self-test suite, and the controls that prove it can fail

A green test suite is not evidence. Two things make it evidence: (1) every check compares two
*independently computed* quantities rather than restating a definition, and (2) a set of **negative
controls** shows that deliberately breaking the code makes the checks go red. Without the second
half, a passing suite may just be a suite that cannot fail.

The controls below each reintroduce a bug that is genuinely easy to write - dropping the constant
term from the Ising Hamiltonian, inverting a gain sign, getting a factor of pi into an angle
conversion, flipping a QUBO diagonal - and confirm the corresponding measurement catches it.

In [80]:
print("negative controls - each one re-breaks a thing we built earlier, and must be caught")
print()
caught = []

# Control 1: the Ising constant term c0 = W_total/2. Dropping it is the common QUBO->Ising slip.
_bad_diag = np.zeros(1 << N)
for b in range(1 << N):
    z = z_signs(b, N)
    _bad_diag[b] = sum(JJ[i, j] * z[i] * z[j] for i in range(N) for j in range(i + 1, N))
_caught1 = abs(_bad_diag.min() - C_DIAG.min()) > 1.0
caught.append(("Ising form: c0 dropped", _caught1,
               "min energy would read %.1f instead of %.1f (off by c0 = %.1f)"
               % (_bad_diag.min(), C_DIAG.min(), C0)))

# Control 2: the flip_gain sign, which was inverted in an earlier draft of this notebook.
def _flip_gain_wrong(i, x, n, wmat):
    same = sum(wmat[i, j] for j in range(n) if wmat[i, j] and x[j] == x[i])
    diff = sum(wmat[i, j] for j in range(n) if wmat[i, j] and x[j] != x[i])
    return diff - same


_x0 = np.zeros(N, int)
while True:
    _g = [_flip_gain_wrong(i, _x0, N, W) for i in range(N)]
    _i = int(np.argmax(_g))
    if _g[_i] <= 0:
        break
    _x0[_i] ^= 1
_caught2 = cut_value(_x0) < c_l - 1e-9
caught.append(("local search: gain sign inverted", _caught2,
               "climbs to cut %.1f, the true optimum is %.1f" % (cut_value(_x0), c_l)))

# Control 3: the pi in the vendor angle convention.
_caught3 = 0.0
for th in np.linspace(-2.0, 2.0, 21):
    _caught3 = max(_caught3, np.abs(zz_ours_matrix(th) - zz_ionq_matrix(th * math.pi)).max())
_caught3 = _caught3 > 1e-6
caught.append(("zz angle: theta x pi instead of theta / pi", _caught3,
               "max |matrix difference| = %.3f" % _caught3))

# Control 4: the QUBO diagonal sign.
_Qbad = to_qubo(EDGES, W)
for i in range(N):
    _Qbad[i, i] = -_Qbad[i, i]
_e4 = max(abs(qubo_energy(_Qbad, np.array([(b >> (N - 1 - k)) & 1 for k in range(N)]))
              + cut_value(np.array([(b >> (N - 1 - k)) & 1 for k in range(N)])))
          for b in range(1 << N))
_caught4 = _e4 > 1e-9
caught.append(("QUBO: diagonal sign flipped", _caught4,
               "max |x'Qx + cut| = %.1f (must be 0)" % _e4))

# Control 5: reading the distribution in the wrong bit order.
_p_wrong = parse_distribution({format(b, "0%db" % N): int(round(pr_noisy[b] * SHOTS))
                               for b in range(1 << N)}, N, order="lsb0")
_caught5 = float(np.abs(_p_wrong / _p_wrong.sum() - pr_noisy).sum()) > 0.5
caught.append(("endianness: msb0 read as lsb0", _caught5,
               "L1 to the true distribution = %.3f"
               % float(np.abs(_p_wrong / _p_wrong.sum() - pr_noisy).sum())))

# Control 6: the initial state. This notebook really had this bug: starting the register in
# |+>^n and then applying the ansatz's own Hadamard layer returns |+>^n -> |0...0>, which is a
# valid circuit and a completely different optimiser. It was caught by the p=1 ratio landing on
# exactly the random mean.
_gam6, _bet6 = LADDER[0][4], LADDER[0][5]
_ins6 = qaoa_instructions(N, EDGES, W, _gam6, _bet6)      # includes its own Hadamard layer
_plus = np.full(1 << N, 2.0 ** (-0.5 * N), complex)
_e6 = expectation_diag(run_ir(_ins6, N, psi=_plus), C_DIAG)   # the historical bug
_e_ok = expectation_diag(run_ir(_ins6, N), C_DIAG)            # |0...0> start, as documented
_caught6 = abs(_e6 - _e_ok) > 1.0
caught.append(("engine: |+> start with the Hadamard layer still applied", _caught6,
               "ratio %.4f instead of %.4f - the register collapses to |0...0>, so the cost "
               "layer does nothing and only the mixer is left" % (ratio_of(_e6), ratio_of(_e_ok))))

for name, got, detail in caught:
    print("  %-44s %s   %s" % (name, "CAUGHT" if got else "MISSED", detail))
print()
n_caught = sum(1 for _, g, _ in caught if g)
print("%d of %d negative controls were caught" % (n_caught, len(caught)))
check("every negative control is caught by the suite", n_caught == len(caught),
      "%d/%d" % (n_caught, len(caught)))

negative controls - each one re-breaks a thing we built earlier, and must be caught

  Ising form: c0 dropped                       CAUGHT   min energy would read -32.0 instead of 4.0 (off by c0 = 36.0)
  local search: gain sign inverted             CAUGHT   climbs to cut 0.0, the true optimum is 68.0
  zz angle: theta x pi instead of theta / pi   CAUGHT   max |matrix difference| = 1.000
  QUBO: diagonal sign flipped                  CAUGHT   max |x'Qx + cut| = 288.0 (must be 0)
  endianness: msb0 read as lsb0                CAUGHT   L1 to the true distribution = 1.443
  engine: |+> start with the Hadamard layer still applied CAUGHT   ratio 0.2534 instead of 0.7208 - the register collapses to |0...0>, so the cost layer does nothing and only the mixer is left

6 of 6 negative controls were caught


True

### The full check register

Every `check()` call made anywhere in the notebook, with the measured value it was given. This is the
list a reviewer should read first: if any entry looks like it could not have failed, that is a
finding worth raising.

In [81]:
ok = sum(1 for _, c, _ in CHECKS if c)
print("%d checks recorded, %d passed, %d failed" % (len(CHECKS), ok, len(CHECKS) - ok))
print()
for i, (name, c, detail) in enumerate(CHECKS, 1):
    print("%3d  %s  %-58s %s" % (i, "PASS" if c else "FAIL", name, detail if not c else ""))
print()
print(BAR)
print("OVERALL: %s   (%d/%d recorded before this cell adds its own two)"
      % ("ALL CHECKS PASSED" if ok == len(CHECKS) else "SUITE FAILED", ok, len(CHECKS)))
print(BAR)
assert ok == len(CHECKS), [c[0] for c in CHECKS if not c[1]]

# A closing self-reference: the suite must be large enough to be worth reading, and every
# entry must carry a name a reviewer can act on.
check("the suite is non-trivial", len(CHECKS) >= 40, "%d checks" % len(CHECKS))
check("no check has an empty name", all(n.strip() for n, _, _ in CHECKS))
print("final tally after the suite's own two checks: %d/%d"
      % (sum(1 for _, c, _ in CHECKS if c), len(CHECKS)))

60 checks recorded, 60 passed, 0 failed

  1  PASS  IR matches dense reference                                 
  2  PASS  dense reference is unitary                                 
  3  PASS  gpi = gpi2 squared up to phase                             
  4  PASS  gpi is unitary                                             
  5  PASS  gpi2 is unitary                                            
  6  PASS  rx is unitary                                              
  7  PASS  engine deterministic                                       
  8  PASS  state stays normalised                                     
  9  PASS  two optimisers agree to 1e-6                               
 10  PASS  QAOA expectation never beats the true ground state         
 11  PASS  flip_gain predicts the true cut change                     
 12  PASS  local search is at least as good as greedy                 
 13  PASS  local search beats every schedule it started from          
 14  PASS  single-flip climb never l

## Part 12 - optional cross-check against the libraries

Everything above is from scratch, which is the point - but from-scratch is not the same as unreviewed.
This Part installs the two libraries a professional team would actually use and asks them the same
questions we asked our own code:

* **Qiskit** rebuilds the same circuit from the same angles and returns a statevector. If our engine
  and Qiskit disagree on any of the 256 amplitudes, one of them is wrong, and it is usually the one
  written by the person making the claim.
* **dimod** converts a QUBO to an Ising model with its own published formulas. If its `ising_to_qubo`
  round-trip disagrees with our `to_qubo`/`ising_form`, our derivation has a sign in it.

The section self-installs, and **skips cleanly if the install is blocked** - in a restricted Colab, in
an offline review environment, or on a machine where you simply do not want 200 MB of dependencies. It
sets `CROSS_CHECK_RAN` either way, so the recap can state honestly which evidence you are looking at.
Nothing in Parts 1-11 depends on it.

In [82]:
# Each referee is tracked separately. If only one library is available, half the cross-check
# still runs - collapsing both onto the availability of the rarer one would throw away evidence.
_HAVE_QISKIT = _HAVE_DIMOD = False
try:
    import qiskit                                # noqa: F401
    _HAVE_QISKIT = True
except ImportError:
    pass
try:
    import dimod                                 # noqa: F401
    _HAVE_DIMOD = True
except ImportError:
    pass

if not (_HAVE_QISKIT and _HAVE_DIMOD):
    try:
        get_ipython().run_line_magic(            # noqa: F821
            "pip", "install -q --quiet qiskit dimod")
        import qiskit, dimod                     # noqa: F401,E402
        _HAVE_QISKIT = _HAVE_DIMOD = True
    except Exception as _e:                      # noqa: BLE001 - the skip is the expected path
        print("optional install did not complete: %s" % _e)
        print("  Parts 1-11 are unaffected; they are self-contained by design.")

CROSS_CHECK_RAN = _HAVE_QISKIT and _HAVE_DIMOD
print("qiskit available : %s%s" % (_HAVE_QISKIT, ("  (v%s)" % qiskit.__version__) if _HAVE_QISKIT else ""))
print("dimod available  : %s%s" % (_HAVE_DIMOD, ("  (v%s)" % dimod.__version__) if _HAVE_DIMOD else ""))
print("CROSS_CHECK_RAN  : %s  %s" % (CROSS_CHECK_RAN,
      "(both referees present)" if CROSS_CHECK_RAN else "(at least one referee unavailable)"))

qiskit available : True  (v2.6.2)
dimod available  : True  (v0.12.23)
CROSS_CHECK_RAN  : True  (both referees present)


In [83]:
if _HAVE_QISKIT:
    from qiskit.quantum_info import Statevector

    gam_x, bet_x = LADDER[-1][4], LADDER[-1][5]
    qc_x = ir_to_qiskit(qaoa_instructions(N, EDGES, W, gam_x, bet_x), N)
    sv_x = Statevector(qc_x.remove_final_measurements(inplace=False))
    p_raw = np.abs(np.asarray(sv_x.data)) ** 2
    p_ours = probabilities(run_ir(qaoa_instructions(N, EDGES, W, gam_x, bet_x), N))

    # Qiskit indexes amplitudes with q0 as the LEAST significant bit; this notebook (and IonQ's
    # probability array) use q0 as the MOST significant one. The two distributions below are the
    # same physical state in two endiannesses - which is exactly the trap Part 8 warned about,
    # now caught by an independent library instead of by us.
    p_qiskit = reverse_bits(p_raw, N)
    l1_raw = float(np.abs(p_raw - p_ours).sum())
    print("comparing without fixing the bit order: L1 = %.3e  <- this is the mistake, not a result"
          % l1_raw)
    check("the endianness mismatch is real, not a modelling difference", l1_raw > 1.0)

    l1 = float(np.abs(p_qiskit - p_ours).sum())
    worst = float(np.abs(p_qiskit - p_ours).max())
    print("our statevector vs Qiskit Statevector (q0 reversed to our convention), p=%d:"
          % LADDER[-1][0])
    print("  max |difference| over %d probabilities : %.3e" % (1 << N, worst))
    print("  L1 distance                            : %.3e" % l1)
    print("  top state under our engine : %s  (cut %.1f)"
          % (show(bits_of(int(np.argmax(p_ours)), N)), cut_value(bits_of(int(np.argmax(p_ours)), N))))
    print("  top state under Qiskit     : %s  (cut %.1f)"
          % (show(bits_of(int(np.argmax(p_qiskit)), N)), cut_value(bits_of(int(np.argmax(p_qiskit)), N))))
    check("Qiskit reproduces our statevector to 1e-10", l1 < 1e-10)
    check("both engines agree on which schedule is most likely",
          int(np.argmax(p_ours)) == int(np.argmax(p_qiskit)))
    check("Qiskit's own expectation of the cost operator matches ours",
          abs(float(np.dot(p_qiskit, C_DIAG)) - E_ideal) < 1e-10)
    check("reading Qiskit in the wrong bit order would have been caught, not absorbed",
          l1_raw > 100.0 * l1)
else:
    print("Qiskit cross-check skipped - library unavailable. Our engine is still checked against the")
    print("independently built dense matrix in Part 4, which is the same test with a different referee.")

comparing without fixing the bit order: L1 = 1.627e+00  <- this is the mistake, not a result
our statevector vs Qiskit Statevector (q0 reversed to our convention), p=4:
  max |difference| over 256 probabilities : 1.110e-16
  L1 distance                            : 6.829e-16
  top state under our engine : A=[Charlie,Delta,Hotel] B=[Alpha,Bravo,Echo,Foxtrot,Golf]  (cut 68.0)
  top state under Qiskit     : A=[Charlie,Delta,Hotel] B=[Alpha,Bravo,Echo,Foxtrot,Golf]  (cut 68.0)


In [84]:
if _HAVE_DIMOD:
    # dimod 0.12 exposes the conversion as BinaryQuadraticModel.from_qubo(...).to_ising().
    # We compare SPECTRA rather than (J, h, offset) triples: two different splittings of the same
    # Hamiltonian are legitimate, but the diagonal in the z basis is not a matter of taste.
    Q_lin = {}
    for i in range(N):
        Q_lin[(i, i)] = float(-np.sum(W[i]))
    for i, j in EDGES:
        Q_lin[(i, j)] = 2.0 * float(W[i, j])
    bqm = dimod.BinaryQuadraticModel.from_qubo(Q_lin, offset=0.0)
    h_d, J_d, off_d = bqm.to_ising()
    our_c0, our_J, our_h = ising_form(N, EDGES, W)

    diag_d = np.full(1 << N, float(off_d))
    for b in range(1 << N):
        z = z_signs(b, N)
        diag_d[b] += sum(float(h_d.get(i, 0.0)) * z[i] for i in range(N))
        diag_d[b] += sum(float(v) * z[i] * z[j] for (i, j), v in J_d.items())
    # x'Qx = contention - W_total, so dimod's Ising diagonal must equal C_DIAG - W_TOTAL.
    dspec = float(np.abs(diag_d - (C_DIAG - W_TOTAL)).max())
    print("dimod to_ising() spectrum vs our cost operator, %d basis states:" % (1 << N))
    print("  max |dimod diagonal - (contention - W_total)| = %.3e" % dspec)
    check("dimod and our derivation give the same Hamiltonian spectrum", dspec < 1e-9)

    jd = {(min(k[0], k[1]), max(k[0], k[1])): float(v) for k, v in J_d.items()}
    dJ = max(abs(jd.get((i, j), 0.0) - our_J[i, j]) for i, j in EDGES)
    dh = max(abs(float(h_d.get(i, 0.0)) - our_h[i]) for i in range(N))
    doff = abs(float(off_d) - our_c0)
    print("  term by term: max |dJ| %.3e   max |dh| %.3e   d(offset) %.6f" % (dJ, dh, doff))
    print("  (a non-zero dh here would mean dimod puts the degree terms in the fields and we put")
    print("     them in the QUBO diagonal - the spectra still match, so only the bookkeeping differs)")
    check("our couplings are dimod's couplings", dJ < 1e-12)
    # Ours is the CONTENTION operator (offset +36); dimod's is x'Qx (offset -36). The difference
    # is exactly W_total, which is the whole relationship between the two objectives.
    print("  offset: dimod %.6f, ours %.6f, difference %.6f = W_total %.1f"
          % (off_d, our_c0, doff, W_TOTAL))
    check("the offset difference is exactly W_total, i.e. cut vs contention",
          abs(doff - W_TOTAL) < 1e-9, "doff %.6f, W_total %.1f" % (doff, W_TOTAL))

    # dimod.ising_to_qubo takes (h, J, offset) and returns (Q, offset). Comparing dict entries
    # across two libraries is fragile - key order alone can differ - so compare the FUNCTION.
    back_q, back_off = dimod.ising_to_qubo(dict(h_d), dict(J_d), float(off_d))
    Q_ours = to_qubo(EDGES, W)
    dq = 0.0
    for b in range(1 << N):
        x = np.array([(b >> (N - 1 - k)) & 1 for k in range(N)], float)
        e_back = sum(v * x[i] * x[j] for (i, j), v in back_q.items()) + float(back_off)
        dq = max(dq, abs(e_back - float(x @ Q_ours @ x)))
    print("  Ising -> QUBO round trip: max |energy difference| over %d schedules = %.3e"
          % (1 << N, dq))
    check("the QUBO -> Ising -> QUBO round trip returns our QUBO as a function", dq < 1e-9)
    check("the round trip is not being rescued by an offset we ignored",
          abs(float(back_off)) < 1e-9)

    dm_best = dimod.ExactSolver().sample_qubo(Q_lin).first
    print("  dimod ExactSolver best QUBO energy %.1f -> cut %.1f  (ours %.1f)"
          % (dm_best.energy, -dm_best.energy, OPT))
    check("dimod's brute-force sampler finds our optimum", abs(-dm_best.energy - OPT) < 1e-9)
else:
    print("dimod cross-check skipped - library unavailable.")
    print("Our QUBO was still verified against brute force directly: max |x'Qx + cut| = 0.0 over")
    print("all %d schedules, which is a stronger test than agreeing with one library." % (1 << N))

dimod to_ising() spectrum vs our cost operator, 256 basis states:
  max |dimod diagonal - (contention - W_total)| = 0.000e+00
  term by term: max |dJ| 0.000e+00   max |dh| 0.000e+00   d(offset) 72.000000
  (a non-zero dh here would mean dimod puts the degree terms in the fields and we put
     them in the QUBO diagonal - the spectra still match, so only the bookkeeping differs)
  offset: dimod -36.000000, ours 36.000000, difference 72.000000 = W_total 72.0
  Ising -> QUBO round trip: max |energy difference| over 256 schedules = 0.000e+00
  dimod ExactSolver best QUBO energy -68.0 -> cut 68.0  (ours 68.0)


In [85]:
fig, ax = plt.subplots(figsize=(7.4, 4.4))
labels = ["our engine vs a dense\nmatrix we built ourselves",
          "our engine vs\nQiskit Statevector",
          "our Ising form vs\ndimod's Hamiltonian",
          "our optimum vs\ndimod ExactSolver"]
# The dimod bar is the SPECTRUM difference. The two formulations legitimately disagree by W_total
# in the constant term, and putting that on this chart would read as "their code is off by 72"
# when it is a cut-vs-contention bookkeeping choice we verified separately.
vals = [3.0e-16, l1 if _HAVE_QISKIT else 0.0, dspec if _HAVE_DIMOD else 0.0,
        abs(-dm_best.energy - OPT) if _HAVE_DIMOD else 0.0]
ran = [True, bool(_HAVE_QISKIT), bool(_HAVE_DIMOD), bool(_HAVE_DIMOD)]
cols = ["#2ca02c" if r else "0.7" for r in ran]
ax.barh(labels, [max(v, 1e-18) for v in vals], color=cols, edgecolor="k", lw=0.6)
ax.set_xscale("log")
ax.set_xlim(1e-19, 1e-2)
ax.set_xlabel("largest disagreement with the independent reference (log scale)")
ax.set_title("Independent referees: how far our from-scratch code is from somebody else's", fontsize=9)
for i, (v, r) in enumerate(zip(vals, ran)):
    ax.text(max(v, 1e-18) * 1.6, i, ("%.1e" % v) if r else "not run", va="center", fontsize=8)
ax.invert_yaxis()
plt.tight_layout(); plt.show(); plt.close(fig)
print("CROSS_CHECK_RAN = %s" % CROSS_CHECK_RAN)
print("the grey bars are honest absences, not passes. The dense-matrix check in Part 4 needs no")
print("library and is always live, which is why it is the leftmost bar.")
if _HAVE_DIMOD:
    print("deliberately NOT on this chart: dimod's constant differs from ours by %.1f = W_total."
          % doff)
    print("That is the cut-vs-contention offset. It is expected, and the cell above checks it.")

CROSS_CHECK_RAN = True
the grey bars are honest absences, not passes. The dense-matrix check in Part 4 needs no
library and is always live, which is why it is the leftmost bar.
deliberately NOT on this chart: dimod's constant differs from ours by 72.0 = W_total.
That is the cut-vs-contention offset. It is expected, and the cell above checks it.


## Part 13 - what this actually taught, and what to do with it

### The three-sentence version

QAOA on an 8-node crew-contention instance reaches 0.909 of the known optimum noiselessly and 0.865
under the published Forte error rates, while a multi-start single-flip climb reaches 1.000 in under
10 ms for free - so on this problem the quantum answer is *worse, slower and $0.70*, and the honest
recommendation is not to buy it yet. The circuit is small enough that the machine's real limitation
is not the noise floor but the **ansatz**: expected faults per execution are 0.27, and the ratio only
halves past a 2q error rate an order of magnitude worse than shipping hardware. And the two things
that would change the verdict - larger, locally-constrained instances where the classical climb
stalls, and native constraint handling - are exactly the two things this notebook could not test,
which is why they are listed as unverified rather than as prospects.

### What transfers to a real engagement

| What we did here | What the customer version looks like |
|---|---|
| 8 crews, 14 conflict pairs, solved exactly by brute force | 200-2,000 crews: no exact reference exists, so the "ratio" column becomes a gap against a solver bound |
| One instance, one weight matrix from a spreadsheet | The weights *are* the engagement. Getting `w_ij` from operations is the hard 60% of the project |
| Cardinality penalty with a measured onset | Real constraints are shift-legality, driver-hours, union rules - each one a penalty term whose strength has to be swept, not assumed |
| `contention = W_total - cut`, so a cut is a money number | Keep this all the way through. The moment the objective becomes an abstract "energy" the customer stops being able to audit you |
| A replay harness because we had no QPU token | Every real pilot has one of these. The analysis code must not care where the distribution came from |
| Negative controls in Part 11 | This is the deliverable that survives contact with a reviewer. A green suite with no controls gets picked apart |

### Exercises, in order of how much they will teach you

1. **Break the instance, not the code.** Add a 9th crew and three conflicts, then re-run everything.
   Which numbers move, and which of your checks would have caught a mistake in the new weight matrix?
   (The answer is `max |x'Qx + cut|`; the ratio checks will not save you.)
2. **Make the classical baseline earn its place.** Replace single-flip local search with a *swap*
   neighbourhood (move two crews at once) and with a 2-opt variant. On this instance both still find
   68 - so find the smallest instance where single-flip stalls, and report it.
3. **Force the balance constraint into the circuit.** Instead of penalising, restrict the mixer to
   the balanced subspace (a `rx` layer that only rotates between states with four 0s and four 1s).
   Measure what that does to `P(optimal rota)` at p=4 and to the gate count.
4. **Reproduce the noise result with the vendor's model.** `qiskit-ionq` accepts
   `noise_model="forte-1"`, which is calibration-derived rather than a fixed single-qubit rate. Run
   our p=4 circuit through it and compare against our Pauli-twirl number. Expect the vendor model to
   be *worse*, and find out which gate it blames.
5. **The one that matters for a job conversation:** our p-ladder is non-monotone in the wrong place
   (p=1..3 crawl, p=4 jumps). Reproduce it, then find out whether that is the parameter landscape or
   our optimiser giving up. The two explanations have completely different remedies.

### Where this goes in the real world

IonQ's own positioning has moved from "one day" to "useful now, on the right problems", and the
honest version of that claim is narrower than the marketing version. What this notebook demonstrates
is the *shape* of a defensible answer: a business objective that stays in money units end to end, a
QUBO and Ising form verified against each other and against brute force, a circuit expressed in the
target machine's real gate set with its real limitations (no global entangling gate, so depth is
priced per edge), a noise model labelled as a floor rather than a forecast, a cost model that says
UNKNOWN where the vendor does, and a classical baseline that is allowed to win.

The parts a hiring manager should take from this: the sign errors, the factor-of-pi, the inverted
hill climb and the wrong initial state are all **real bugs that were in this notebook's earlier
drafts**, each caught by a measurement rather than by a reading, and each now guarded by a negative
control. That is the job. The physics is the easy part.